# Task 4: 5-Fold SGKF Nightmare-Hardened 2:1 Vision Trainer (Kaggle T4 × 2 Parallel & H100 Ready)

Same corruption-hardened **V3 2:1 native aspect-ratio pipeline** (matching Colab `cnxv3` & `dinov3l` SGKF splits 100%) — configured for **Kaggle `GPU T4 x2` (Split Folds `[1, 2, 3, 4]`)** and **H100 Turbo**!

### Completed Colab Reference Models
- `cnxv3` (`convnext_base.dinov3_lvd1689m` @ `192x384`): `OOF_W6 F1 = 0.98878` | `LL = 0.05356` | `ExtHard F1 = 0.95465`
- `dinov3l` (`vit_large_patch16_dinov3.lvd1689m` @ `192x384`): `OOF_W6 F1 = 0.99166` | `LL = 0.04802` | `ExtHard F1 = 0.96293`

### Kaggle Accelerator Guide & Expected Runtime
- **Recommended Now — `GPU T4 x2` (`16 GB × 2`):**
  - Default **`MODELS_TO_RUN = ["eva02l"]`**, **`FOLDS_TO_RUN = [1, 2, 3, 4]`** runs 2 folds on `cuda:0` + 2 folds on `cuda:1` in parallel (~`2 × 4.6h ≈ 9.2h` < Kaggle's 12h limit!), pairing with **Colab Fold 0** for a complete 5-fold `eva02l`!
  - Follow-up Session 2: set **`MODELS_TO_RUN = ["siglip_l16"]`**, **`FOLDS_TO_RUN = None`** (all 5 folds ~`10h < 12h` on `T4 x2`).
- **Why NOT `TPU v5e-8`:** PyTorch XLA requires rewriting dynamic CUDA masking/quantile/eval ops and suffers heavy XLA graph recompilation stalls on dynamic Nightmare augmentations.
- **If `GPU H100` (`80 GB`) becomes available:** automatically disables `grad_ckpt`, uses `bfloat16` + `TF32` + `bs=48` (`~1.5–1.8h` for all 5 folds).

### 4-Step Quick Start on Kaggle
1. **Notebook Options (right sidebar):**
   - **Accelerator:** **`GPU T4 x2`**
   - **Internet:** **`ON`** (required to verify `timm>=1.0.22` and download pretrained weights)
2. **Data Input (Zero Upload!):** Click **`+ Add Input`** → select Competition **`player-intelligence-system-cpe-342-ml`** (Cell 2 auto-detects `/kaggle/input/**`, symlinks images using 0 GB working quota, normalizes CSVs, and restores the exact verified 31,546-row `folds_sgkf.npy`). *(Also auto-imports any `folds_*` attached from previous runs!)*
3. Verify **`MODELS_TO_RUN = ["eva02l"]`** and **`FOLDS_TO_RUN = [1, 2, 3, 4]`** in **Cell 1**.
4. Click **Save & Run All (Commit)**. When done, download `t4_hard_folds_eva02l.zip` / `t4_hard_bundles.zip` from `/kaggle/working/t4_hard_outputs/` and merge with Colab's `fold0.npz`!

In [ ]:
# Cell 1: Dependencies (timm >= 1.0.22), Kaggle/GPU Precision, H100 Turbo / T4x2 Config & Resume Scan
import importlib, importlib.metadata, os, shutil, subprocess, sys

def parse_ver(v: str):
    parts = []
    for tok in v.split(".")[:3]:
        digits = "".join(ch for ch in tok if ch.isdigit())
        parts.append(int(digits) if digits else 0)
    return tuple(parts)

need_install = []
try:
    cur_timm = importlib.metadata.version("timm")
    if parse_ver(cur_timm) < (1, 0, 22):
        print(f"Upgrading timm {cur_timm} -> >=1.0.22 (required for DINOv3 & SigLIP-SO400M)...")
        need_install.append("timm>=1.0.22")
except importlib.metadata.PackageNotFoundError:
    need_install.append("timm>=1.0.22")

for pkg in ("scikit-learn", "pandas", "numpy", "pillow"):
    try:
        importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError:
        need_install.append(pkg)

if need_install:
    had_stale_timm = "timm" in sys.modules
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-qU", *need_install])
    importlib.invalidate_caches()
    if had_stale_timm:
        raise RuntimeError("Upgraded timm in-place; click Restart Session once and Run All!")

import numpy as np
import pandas as pd
import timm
import torch

print(f"Verified versions -> timm=={timm.__version__} | torch=={torch.__version__} | python=={sys.version.split()[0]}")
assert parse_ver(timm.__version__) >= (1, 0, 22), f"Restart session once! Loaded stale timm {timm.__version__}"

# --- Run Configuration ---
# On Kaggle T4 x2 (<12h limit): run MODELS_TO_RUN=["eva02l"], FOLDS_TO_RUN=[1, 2, 3, 4] alongside Colab Fold 0.
# Follow-up T4 x2 session: MODELS_TO_RUN=["siglip_l16"], FOLDS_TO_RUN=None (all 5 folds ~10h).
MODELS_TO_RUN = ["eva02l"]
FOLDS_TO_RUN = [1, 2, 3, 4]        # Fold 0 is finishing on Colab! Set to None for all 5 folds
FREEZE_T4_LOWER = True              # Freeze lowest ViT blocks only; keeps patch_embed/head/attn_pool trainable
FREEZE_VIT_BLOCKS = None            # None -> model default (eva02l/dinov3l=6, siglip_l16=8, siglip_so400m=10)
EPOCHS_OVERRIDE = None              # None -> calibrated defaults (eva02l=4, siglip_so400m=4, dinov3l=5, siglip_l16=5, cnxv3=6)
MAX_FOLDS = 5
ENABLE_DUAL_GPU_PARALLEL = True     # Auto-parallelizes folds across 2 GPUs on Kaggle T4 x2
DISABLE_GRAD_CKPT_HIGH_VRAM = True  # Auto-disables grad checkpointing on >=60 GB VRAM (H100/A100-80GB) for ~35% speedup
P_NIGHTMARE = 0.50                  # Balanced 50% clean/flip + 50% nightmare corruptions
P_NORM_VIEW = 0.25                  # Probability of interior Gray-World contrast stretch during training
USE_DRIVE = False                   # Colab-only fallback
SEED = 42

def select_amp_dtype(device: torch.device):
    """Use bfloat16 strictly on Ampere+ (cc >= 8.0); Turing T4 (sm_75) uses float16 + GradScaler."""
    if device.type != "cuda":
        return None, False
    dev_idx = 0 if device.index is None else int(device.index)
    cc_major, _ = torch.cuda.get_device_capability(dev_idx)
    if cc_major >= 8 and torch.cuda.is_bf16_supported():
        return torch.bfloat16, False
    return torch.float16, True

IS_KAGGLE = os.path.isdir("/kaggle/working")
IS_COLAB = os.path.isdir("/content") and not IS_KAGGLE
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
AMP_DTYPE, USE_SCALER = select_amp_dtype(DEVICE)
N_GPUS = torch.cuda.device_count() if DEVICE.type == "cuda" else 0
NUM_WORKERS = min(4, max(2, (os.cpu_count() or 2) // 2)) if DEVICE.type == "cuda" else 0

H100_TURBO = False
if DEVICE.type == "cuda":
    cc_major, _ = torch.cuda.get_device_capability(0)
    torch.backends.cudnn.benchmark = (cc_major >= 8)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")
    for gi in range(N_GPUS):
        gp = torch.cuda.get_device_properties(gi)
        gvram = gp.total_memory / (1024**3)
        g_amp, _ = select_amp_dtype(torch.device(f"cuda:{gi}"))
        if gvram >= 60.0:
            H100_TURBO = True
        print(
            f"GPU[{gi}]: {gp.name} ({gvram:.1f} GB VRAM) | cc={gp.major}.{gp.minor} | "
            f"amp={g_amp} | cpus={os.cpu_count()}"
        )
    print(f"Runtime Mode -> N_GPUS={N_GPUS} | FOLDS_TO_RUN={FOLDS_TO_RUN} | H100_TURBO={H100_TURBO and DISABLE_GRAD_CKPT_HIGH_VRAM} | DUAL_GPU_PARALLEL={ENABLE_DUAL_GPU_PARALLEL and N_GPUS >= 2} | NUM_WORKERS={NUM_WORKERS}")
else:
    print(f"WARNING: Running on {DEVICE} (no CUDA GPU detected!)")

if IS_KAGGLE:
    OUT_DIR = "/kaggle/working/t4_hard_outputs"
elif IS_COLAB and USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT_DIR = "/content/drive/MyDrive/t4_hard_outputs"
    except Exception as exc:
        print(f"Drive mount skipped ({exc}); using /content/t4_hard_outputs")
        OUT_DIR = "/content/t4_hard_outputs"
elif IS_COLAB:
    OUT_DIR = "/content/t4_hard_outputs"
else:
    OUT_DIR = os.path.abspath("./t4_hard_outputs")

os.makedirs(OUT_DIR, exist_ok=True)

# Import any prior completed folds/bundles attached via /kaggle/input (supports multi-session resume)
if IS_KAGGLE and os.path.isdir("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
        bname = os.path.basename(root)
        if bname.startswith("folds_"):
            mname = bname[len("folds_"):]
            if mname in MODELS_TO_RUN:
                dst_fd = os.path.join(OUT_DIR, bname)
                os.makedirs(dst_fd, exist_ok=True)
                for fn in files:
                    if fn.startswith("fold") and fn.endswith(".npz"):
                        dst_p = os.path.join(dst_fd, fn)
                        if not os.path.isfile(dst_p):
                            shutil.copy2(os.path.join(root, fn), dst_p)
                            print(f"Imported prior resume checkpoint: {bname}/{fn}")

_free_gb = shutil.disk_usage(OUT_DIR).free / (1024**3)
print(f"Outputs & zips will be saved to: {OUT_DIR} (free space: {_free_gb:.2f} GB)")
for _mk in MODELS_TO_RUN:
    _fd = os.path.join(OUT_DIR, f"folds_{_mk}")
    _done = sorted(f for f in os.listdir(_fd) if f.startswith("fold") and f.endswith(".npz")) if os.path.isdir(_fd) else []
    _ckpt = sorted(f for f in os.listdir(_fd) if f.endswith("_ckpt.pt")) if os.path.isdir(_fd) else []
    print(f"  Existing [{_mk}]: completed folds={_done} | epoch_ckpts={_ckpt}")

In [ ]:
# Cell 2: Smart Dataset Resolver (Zero-Upload Kaggle Competition Input + Pack/SHA-256 Support + Verified SGKF Folds)
import base64, glob, hashlib, os, shutil, subprocess, zipfile, zlib
import numpy as np
import pandas as pd

if "IS_KAGGLE" not in globals():
    IS_KAGGLE = os.path.isdir("/kaggle/working")
if "IS_COLAB" not in globals():
    IS_COLAB = os.path.isdir("/content") and not IS_KAGGLE

DATA_DIR = "/kaggle/working/t4_data" if IS_KAGGLE else ("/content/t4_data" if IS_COLAB else os.path.abspath("./t4_data"))

EXPECTED_TAR_BYTES = 768022016
EXPECTED_TAR_SHA256 = "d7316deeb95c78ff230229c84f5a2ffa0cd86874a2753405ac9917ccc68a3a44"
EXPECTED_NUM_PARTS = 37
EXPECTED_STD_PART_BYTES = 20971520
EXPECTED_LAST_PART_BYTES = 13047296

# Exact 5-fold SGKF split array (31,546 rows) packed in base-5 + zlib + base64; matches cnxv3/dinov3l OOF row-for-row
EXPECTED_FOLDS_N = 31546
EXPECTED_FOLDS_SHA256 = "e392c7d495259299"
EMBEDDED_FOLDS_B64 = (
    "eNoNl/tD8WwYxx+KkeZ8yuhdYXOYQwkP6mFjc24aCpXNYZhpciqqP/7t5/un676u7+FzNJ8sLdDI4e1rRP0YZhaQqatDPF22h+5ZWCXlfNyV+THPawqHOoVtAfkQmLDh7nmHPFu7ChvgIbmrqOxk3HG6+qCV2uWTHanuAX04a5ycWRGVsCMhzDKayluV+gVh3Qv5y9jqUqu7X06au+XtYlZFY4FKS4Of92Nvdakd63JnnZX1v7XRBKBKGiElcWEuh/9VD2Bg6lgLZKw5qLWmpZrLjRa+/uplkY5Fy8qxdLEmdDKxT8SudfS9vPEYerD2j5f3vuziFLfZEU+obH5kiTRHehdm60mz2K2x6SJN3t4djOsj+EMWOIDlIw7MXOzhGw6gPzV4xN0yU3rJ3SuCLv7vgE0WIr03WzEEhldxvoeQSk9nA0SpDX/3Wa/d+NkvLFi2qnUo8kKdifEfMOp2irolHjoGCiWYZo5yVzLWdwDqQ8hLDs+/r/+Tgk+8SAMt68vYUpQGy1jH1dkIsbiNKWpaYNQkLm8+3MMjeHHDtqhizDnf39zvFc2i7t9O+Oe7DOU3I8fddUExN9UpVcbd7M9i/kKgpDRAkdG9gsvUQNGwBwiRacyA0XneubUUc/RKkI+Sse9HERzMEFrdAmUvmPKrMHN9XeRJraJ+uigFNqYrKfVYpVJkzG8ModfNkVb5NDLD5OsYFmlRg+vRcXqIaeLHaTWb/pI9F2PDMUMpyZPgVPu3WOdBUiYFbZpiPXIJjp58J//ON77vVgwpNSx/yeP5KXtvaqHXMDkCgfbCtDRdnL5sljw4EBUX+NTDl1XI5quhIUknW0peVDSd09QUjc+qOjhJfoagl4FTFZq4h8pKj7vhL/FNpAPMnCFUWTdWUj1U2XFpMFKvQfn8Quqf9SLH2jl4c99yU/OAamVGZkDvlus+6Pax9KfOsSwGerwyy42JGUPF4VHwYy8xpwtolhd6IcSkQUKLHY9MPJtQV6VVluPWFRPR+pYAFmD3RNyz0YZv+oRNRm6QvQdEC2ZolOUcF7pD/L1WBmSpw1Erb23J/ZF9opGmkbjdl5hruvXz7evhtHWNOToZfaY1+Za5A30VWTw1rJk3XVpA1ooMTbTe9fqckw+Ik/hEZPtpAXRHxeXRso2Mw/DIQMoI5FNGweQWeDnUToao4uWFn/ABKfCNmINUc3JIxe8t4iCgmJ1n8mReji7c9EqJLnvaHuVX8YUJQaSnizFyKfeaVBUXvCglFiIdahv8YGlQc/lv2SIWX/s02HV2g8mRSbR42uSrTUC66xxoC8BnR/CUvozPHDcFuy4VppDRKQth0bhyxDij/hR0uOVZ11SJi+3PC8t7j1+6W/jD5OtRY9WVtyetSaQZk/XLQHsMl4/+MAsKmaWAj7JeOZ4Qk09DsTm/VX0mS6QKFGtv1y/oMcClC461/GPbNW5D5B7wZWPhGEt6DFd68eXokGLdWA6xAzUhSM4NF7cjQYM7STanSIfS5IVW8zKqAWWL2duk/AbKkj38NZUFGoLBrp4NsAgXnzo9I1nk+oFqIGSBz7GXjMO0ooFhkjz2df1Fg+ZL3vbzRYNW1fXPtfgx8rA6ZRsWkSmWq49qNbLHw7sjJh+YaY0MDSx870wLFB10e+IZVK/+eMofFmilHK4CID+ggaqnhSdZnpy7R867TD8IykoXUld6qM/BIXC4vm3dTcs/wMX4hU/Ft2ovaw/cs4F5tvq2Sk9irJ8yaT3soHh/mUBMskdMmDx0V3V79aYZQts7/8rKd44L3ItFs5y1pwi2AtTCjWFq4TX0387WmRda/ZJPle3LJOBbjQOsvmJ002H5I/8ykg28nHsNvQrOUuEHQJQ432Ep9wY7QpHN1e0rrHs7Sz2X5kGzmFXryVf38fa9BxzOPJ1VbeKk0od6+VLGHq6Rd7qprSU3JTOMb0nQ9ThNzjxDnbdeduRDQ0AwzbPDseumgQjGNIFI2w2fc6zYACEwxy5U29R0XRq9KTvVl7BLMKKiThRZ/ernIVcFTJm8O+rmubYjkN885EwTdSqw0nFfKlI4QfrX0TgfwfWiriUQofQdXZOAEaNK8DdXHUeZT0RrkXYvbP0k3SSkfB4X9l7QrrkggYA01/R1DyvrZb107X87KbBYjEU+HFo9uD4rXMVEZiqs9l5E0e2NzjxbstqxpPrAd9eXCP2XXVVJ/k8nlHr+qcZJuPuckFX0Bg/p5iaTLUvyL2Y4O/47AhvOgYqtsMpscx2qqcnaX7DcBc0qNP/WVJTdtJIomWxtQYvsZRyAM1+BhQG0GiOHmL+rg4jxDloU41t5c3lnviR+oF5kc5lgwPEB93ZDL5a1qiz+UZaLzpMmtWhUIFW3CRSPVh+hkszdB9hDwpmja1cpZ53/MZ08U01S9kgM9r6CBYR1q+s32ESZvH5p00b1XSDGUWAUIkYaLdEaxRp5UNT8vFya9OWTy8Kk5qUbe29vjnfV3YDO5FilqcklURiNURZw91cqHsWsJKDqA6ll/NsMCzf5QahWZA+gJ2A0z4u3bFTz0rSejwwM88Xq0QAoQCOPaVl9b/mgYkc0x/pzxMwjKv7r5oxGus6+Wl1ytFkP8fV9BHDxA3ai1RNfhGp1HWlO7yExZbMmimfI6rybWJQ1wRs+CvSiAJdIt8zl55jZM2FMms+iuyDBvc0beCA1Mi0NYYjWTUivTrsClYIgHtV7wTeBlBX6qmKycP03uz/wL6w+NdS8volAtf4Jtbw6wLbxBgRpCGCS5Vcphd3V9jY9cMmIUP3lS69EjfxXRT+TB0ZHMQ2f+PaJhHnFGDhEL2qZLeiqRcqyaRKtt645saTiJFNth+6hF3G85fDm1Guvee+HgqetXuIsXQ6cSU0ebCWUOa5rDraAildv7fGLZNfRWVeRiTV4KZOmNSQ1EgB9bmR53OLi7EL72pZlhRexW9Pq59O7/sWP5gOQ3KwGWcH39HZJZ8cKd9eCh5IteYEjEgqSFfG0ugxaGtnNIj0zcIp8F1iMgGrq+Y6nbUxNLB+fe/OS/ykz46xYD4ltwr9S7FqJ49qAg0yetjkwmGofWlqgwIaSxMfVSfrTcavK6SUFLtb95LHdY/XqO2YswyIrQEPqGqEIMnGK1fZOHCmysqvUwlRqlIxCoD1HtPjA4mUdRGFWkYFuska/xb1RWuHEcnQ459O/Gzm7l0cgysu+sV6TPisUzDAVpkmZt7eP2NSTpvQBEPo3MCj9l/06yvnssru+aK0/YKFQNgM5P11wGlj30KkyS3+4j0YqV4HP75Bfq8/uefLNHx9oiz5zEW9KsAsovJ1w3g1fmMEiWKldk89n3HUIybRtJpTmva1bJjQd+k3nuZqbK5n0RGbwR+xaWshRa5Lhb4LRd5l1+7XawAkauY8+apm6hc06pNGNQsPdGZtUZBLUXnxBbFEFrhGn8wpQIQddCRhMocIBpeWIpRPK2pPxMAxu61duN+kdySAj/4bHCmHT6SVImiglLqSj70feT8gYKvsmXV84YYvUyNs9BqoxJG0bE5edeflPNeIHJ0su9GRQLQ6WWC37yVjy0ljEGMoH3E1pYAt5OZvxQ+ZdHN6PhW8Q1djOtsiySp4FclwbZjWvP49/NnDe5H2RH4LygWVwrkqxATtQnzPmSEHif8ce/snXisA4yBfL5liWe2qg07I6Cc6aCPmGT3jVOgM2KV4+goDgpCfkowK7apxEBrIN9tbOQbs1Z0ZoX+W/E/ZDWUz1Dic/iDS/qYWzGcXoifWMSDmnufhV3RiJ4IvfPitYDMSMi/H5l7H7SpULWOl2UCgHavtIVVV+i/UypCO2+oZ0XHy+kC5OCMNa8h8P0uMIwoNm+HV47UXt+upGQ2Tcn3yWb91Om15poNTWFlOiGN9g6EoupRi+PDqKWAoPk+ejUUEe8nLBcSaGCOQRUR6O0UmQz12emmR9lTvDmsyAOeXft26/a152hakqAQX8eaxZlFCOFsAr8EMZsFpwrOyQChWVSQOuLOK8MC5XU4dzcoPD70BfVlr2Q6evooG64N3cXo5w4LeLQsxwm40GcCHHdyE6uAaAzlauXldhoI9/zw3ldNzsGQH+n7wZsgyI6gxxaRFx04GuYHoEsWPYZPBu73oVziQCO33IPZCC9jRq+wR0IvEG6GLCLreJmdw8NGowIqMdjsUFmFDm5+qCNTDxiTQBVVp/gsXRqRIin99rkT1Oe4hV07rVoSt1WiCsxUDNIDpZR3xhL0rbySY8eamV1Bxaj/P6WikHgFzC8cgWkXFwDXHT5CBm+74wdF0jXcaMIyjYgKn71sEG1BkTMEBcnHMCVWuar507zoXqAEcq8B8N0J2GcaYW4W2D4uIM6PD6AdB9VzSlt5tG2k7YtL/o1IdMqnSkqcPZoqeOHiMf/I3dswW47tjPDmO71vnNy9ZnXQSFkmbaii0oYAEZ+MIeS3qlLpe41wcWdQkGMS3PGA+JcFI/gJ4FPDLNTOMTf1aNUXk9d92vpSgX0Of6K8XN/LTvnGpQ9xYtO7Q9tC/zUJwreyjwFg05lIHnL59y0YK3F05+pehIrZiqEHqkRB4j3Bahs/fmlS8mn6UCtU4+3ff08yJ+O7d854BXYCnn+pqIulFBmieeKVpifRwK1vCvew/f1/IIn8V4Mo78nKV3sZXInHBnhe+jRVKD1VOfOuQbMiHuIlff+9BW0oCU9dwVGNhVgW8hrwqUxwEOWMgf1cOMvPcWVwD3wjv43rgyOC3G4gc0Lxl8rP96m8lcB61+S7yYofmEqdmpqxP+rou0kLW0oc4HkUjZcvQXElaOp0+eRaiIUbjrn+2hOAchQDksbxa12qfvI/hwet+1W2prFaZ530rjFRdvZy+s+p8qPCJOwtvT0YWmMfm2JPLbU/TBHNinwSvMLWyA1d4HohHXPWHJ1VH94joLGL/VCXkUnoJEelxhTPzfmKmQCJbkkTt6AX3zWpVU9yzqa68esC2N1bF74uRKDX1kw6eJGdD6p33Rk2qgA0wC5LfsGKbdAzwUAX2bskDkvB0zDwyeXYkFMlLKiV0j8V8Y7Ixkd91jR5b9Qox2bxxNjx8YthLU4q7w9VONlM3IwPgxzX6EpIsR86ui6jfQGAHEvqNG76cAMr99vXKzWP4Y1XCnDWnq32balJooTslaxlFJkrIhosLn7+1fi/lPDuA/QYDV8ZldR+b9UoU8eVZNBhIOBGWV5VxV8wa9RsEQTG0nGIsMrv/zOn8spnys33Rs3GLe7EO6GWBna4ciqg2auc+Y9Dmw8VvWDX6OL7TeZRFDX91V7Nx8V68unKDi0SoE9N2dqzFRBPt0z+0pzrek0kx0p/yXG1HtUQRXcSTo3kTeQPCyXK/ETOrbEQIU9jBzFAJ5cGf9L1TgSYf09WB9EpzYMwlRe3+JVa8EPLBPLarJTlSa3zZQo/CdNyMIywPlfYM0p4I73VsMZgNUbXvLD4VsNFaq0VGNmCuzKho41P7Uy8DDZMA60d/q1dd0lV96Uid22luLaMT5i0y7dHnPrnk5V4+g4DpIq/cgxCFzAlSqMl8cSDm9nyNgzAHYLyP+ggJ/XYgmvSNAwuppnHJujfLvuD8BrZv65bjku0BZt6G41uuQrXtXWzlDaemsBebenI1ybjZtWQO1PhY7jX+q8ceWLB0R6325JW07J7N6FoUtmJ5UPDPShAWeLd4h33Wpkd9ND+5oS7DNot3zStvJab33IyCpugpU0cw1Fkeey1fGe6/4an6DoaGCwAbHyX6E0hhQR3aEOzFlxQ2qY9IRhq8zhM5SjkwDxtryXfWtIP1asc7K5NztPggZOaB/a7kozWQg6+YOZyLx9WkpVCfdxRGZWIaTg6OB9yZcxQfRGSIjhdj2usOi/uv+hOEZFiCozl4fqQniFqQ7x4qshF16HwkXLg7OTUlfRlhJ08gFMZCHpdMBEGRTFtdjgnr2bumYUGq66ZpmAqxYsvaLQ3bPhum8TaoHwKQixXK9ofGvupa26S/OHvIfsqx4KZrqZ6gq9anItcBfU0bqu+OLY7ErK7CoApH6TgpuHVcaTK4ajDET8FuT6qOV6tXxZRzZ9bxjz8c/RNFXrVtsRYbWylAfwhKB2tfG/Sx+K7oxc+EAlDoq+8VwRceKPW2N8gyUZTY0z7UnT31lgs1QfpJWdlP16qcrYLk9DbrAvpJ0xN1UKC2Qi8RUcBaBAi9WjAcQV/DgJLhJ/d79xvNxQqyPLmNDMMxn7XrEEmOz/3EakmW+3SsbQgEmG8p8ngQTQ/V2NYbjYvRJrUj6S9OjT1gPisK47sg9T0H1Q/7AT4gOLUN+/504XqcTys4mYhaDnnhkH2iBKxfFbBDp+PdQWa+lE9taNaZhqO5pC6VobVLgg6n+HjaPHDgRG0rOMcwiazqgwRz5ZXLwAwUn4zJiDjmzv+SZ0Lfnz0sH3kdmyrJrKkWS/EXFmKdE3zvz67qcSZ+S/rJRfCMwZSVpxZCbo4kMcw4uIVusxPQsBwnHZNkVT9dSJX3xcJYdnsexeFqg9GhfQo69/hzLXu6cUF482jYTv4kjxeor/Shi8gwn3g+zIvIRYQE6IDxrvoz4x/z0+X1MAi0MXNZIHUAswNIbNX4+SNk0LrkulFk2EqELyc7+/Ka/BgKxoQrNUZWjRuyAk55ij3K+k6owX6BPqL8m1tPSiybB+6SKSlioYDsrWR1ZrlbSMuMHcMW/JJx8muIoTINjAa5n92iLJmbK52e+O/HVPeH2styzxCVoeOP6m2UvTslJUuzZ+XTooXc3FQpdDU+RwEiGDIzu8MtNIqvii5VxQ+KQO6uPuXyvfiKyyQuqHzE7dfrRhCnA3NJn5EvxcNYMPye2x7FWwX7nE3rq075Z7Fk0pptH8G1lRsjq5T5lwJtbNxkxzKrMZYmJVaqcY11LtTaRWnxOZf+8LmJkRnPdTwRYiJOt/uZO+8D2HztlUZ64EEUrcM/HP3Asqgn/KevQa46zXEc/nwDtZaizlHv7ez/A3/bEnvUyvmEuYcNb7DC0/vEX1okporYgL8eCSQwMZMj8VBJaBZcn/mN4dI7nJbJ5TImosYxzGp68e9YqBrzYUuDzrLZWmIOSkFxo3QWtjGym+6GJyXPaszTXt54jzPzIZHYJXUaf702REHocoIsj4LU49rIIMDiu/f23ueq3Z+5I1++8n/+L+hqV63D8vUZ81+lMZ3ZBeWgRZ49j6adxmp9AHGl/uP00zvN82chLrWwtzOPnJZOieUyY3QjJN85T+Y9wbNKXPfxc323JI4D2nnH3ObW7PUVHyHH4ZPd7W5lkmSd+2TlbNNHwQ73Ls/7TfuJyIG8FQOcU4VdB6Bt4H2OyLu8Z6LytCTkOQgJSTgA7VTj9ixpSEtN0GRU1mtRSUUm4eV1UmDGkys0KAlfWnyIcKWtz8chIRiFrxQCh+pCmv3GheERdqFGRznXL5T+x5KcAVt8jwUNMKxZifVkmy/+1DXyx43cunYuf6y2GwXSfjqPKF4HsKpafoB0q8PD25Pe477p003JGcQVwMsfkUQrtQAGhsAd2lgDIASwwAh4Z5ZBJZD8lFLSYUaYzVT7v7wLT7WQuT7agPIuyDLI/phF6AyfFgnnBvLWfrVPx9pyMX2zf28OzwzJ90ivBn+f3F7lFsafhO3ycRX15jFXHEG2sG6P5XxLZQsaIqmh64cYIflix6H3GHANaEkF2Vcf/jJf6fxzJ37CWsrOoSwtZOlp+15dlmpA7eTIwviXbfP/O1GafM+0ARtQnM7FV/1WtI+vmwMxgWnUjQzf6BprqzZIM+9ZQ0WSu9mKTLta0mwR+Hto+5Dei0baEuS865BheScIDq1meDMOnJqCZwBSkZoN2CoJukanQ7rRFbCorR6DLGsK5laxjP89Jda6hG3uJKSJMfmsDsteUW7KUjjvlZN2dA/yqLd0f8svsJkRMve+7qbIwN9BeWjlD7jayQCuM2CEi/tbj0ct/RgOFzQ2qAXdGwWDUTJAHNWHVR2xtVMKm7iIYvJ1aOodrmP3lydOPnEXlqRjNSjLSBgPTjXJpRdgnHhJ39kDr9Lc2iA2y7/6M4UKsCwx7RU2y0rMSyrCVwNmu17aVgt9t7Niz+vQV7RtNlMzg8mbYUKY1HJsejLsqbx8APsYheMAPgMJMWXjX8NmRyjPZUDbkahxbtGrFv2aGsyUoj6q9U/tKI+REWWJWsmnpeblJafrQL4JIl96AiZueYNUO9R/fxdE1wMXfl9OmJbW/Pme91U+7MpC57/BmuLDXLDXlyXNkQjuz/HKgwiMxOtQ53sAJ0AbCqIpHJkGoNVsLAefaCM3fEZOPcnTrh/Q1zy4J5MfpBI55coysHCQOR06GGuKLZfy0fh3jTu1BgLSl6RjxkzLyShKxly91wesFChYnv/sdJlXT33HtozNPx65avnxY1Q9el+96SV88Hl38nARTn6JrYKEhYwgUrUFAQjZz3b/FHjm6+0WKxSnwzdvgF9ZinedtsCnBMy3rKzDOUOgzYl5uHn/JGrnIDWbySOADqr89m1T5wtmeeK8ZsDM2som6iS+dxdtXFN6SlpkXjGOtO5BOoJ69ke+sr6PBdGUMMOMg1TJpUr2tl+ahVeaJds8QM7KxefcLdbYWuWYD1qQ3H1WV+9dR7TlDaV9FO3CvZmEIr24cg5WSGGnVCiLLfLzJHusEC/N5uqwvmpXsQBuI3AO7WYzD1YS0sODwlBlexU63oKuBrAH3K185K4e87tHwj5f1DDTDSYxYKbSTBkrldxBge6BLZ+wv/TeoejrE1D4Od5vbzxqitMQvV7bf53eTX96KF10tgPujNQ/Vj9rOX75ccJmswDQZSjEGS3780IW+IS5tBlZzr25AmJGVjpk49z0lEz0ArJcTpuUxTKq6QtfN9U4iBuyYSNlDhbtq+huHzfppqjnMuQw6y3ElsIh/KpCDc9cWMnGgEJkS6tvnuTXO1cWpD17Fi2GtwaYG+cnDpGvA3mWEW7dTv2bHDUum6cFzdtmv/e+m53muPHYTU/k3vx3cau6ak5YdUaYwS70RKM0c4NObHlXg6JCyP86vCdJO6+8SyH2aj9Mlp1oT+JYBXXWkvzG9YBGVtRmO52SCpxMyTRIwMNT/LXwp0pyy82YZad34dKbE0zOIu3n+PCIEEq7XMoy5Xz8vCNqIyTlZocrbDjaeXeTHNhhZJ3vai9uKv+pfiSAf/4F0yOWEBPoKgplGDKZ1VdRg182OGA+ftpKcdrRSFHbQ3+kkavotKukP97hFTKCAFH39UgBr9+K++7r8g6qfsE2YVGS1XNEcYazBpGVUlzEDpDVGe57RmUDeGkMowyQ378pX6iL76f6ZwHXmpOFY/PsBCuemnGprOP8AAnskHt6Z9XmSdpCHhOEzeosF+n8EL0n+eiIOtbR9p/Y3gTl+747jGmL7NOYj/WsGinbLCk2/Ms9Dja0t+VTmMTPDylJp/0xkPkz6W9LxxIGhQFkg57ICsTQKRYArk4r8k2oT+lsWdO8fkuWsYGfO6iJJCyk6uE6SJyMHagBU/Eqz2Ckkxh96zg2hIs2dBl+tgHSVnyFAT1tsL4xv8UjuHhk/7DXp3dxZ8PW/SzI5fQunR+xY6b5diTv9XOoJGNlGkjzXZt2XkZfq5YKUN8VP2e0A6sG7PY2o630Rj8jgcihO3G5+gPvIIIVH3bo3OXuBrqEnPafTzDXBiTNMASNf5qavELuqWFmzUztJWhFb/rKO6Nqpo2Z8kqb4FtQFAlPMzVrImdI7uhm6WVX2yjk9CZvya6UOxXdA6v3DnjrD7ujFHhSDZiCCmj6Vsjxby5G8o7CtKE/aWKwMf9pr1ABAcD6wkzASMj9zVHwsgsjMFL7x3+Ux1MhTuMy/HdyFcc035CSfzyaDB2NOG5AxNPQolm+CJmBIIVvVKcW/rEIwxfCd4kBfN1/kd+pj0oUku0rOMLfWV8JcziILqSp2A7rF0oZnRkaPRPBgCEEL+mMPi8ynuePAbyRPasMwXmLp/95B03+TvNrUc9gySdxEQuh1ds8HuFLxiLlC6pYWWgcYBweawKsb2NYQSB0Kb6CCtUZhIQPwyVEsw1OsP73B58BKIbbK1pGeYmrltPYgezJBrP6qA5SNLsbe0Py6NPSzVmb5nfas4z899AHLaqQ+6UOfMi83Tt3nau0QHBdevvgQpsrGJmd+CRv4SB9OfFXDoIS8y1oZQjak3hNYsoExIL2AzKfzm6ES5PMfKhLgVmPE0hmjxOXFzRSx8ziXeg1df9lhHBkII7FmAbro/djl5VTKUsD0Zf4EfIENYklt8W+LlzTDtAAXrkDNlaYzkgvbknHefgPAZbVziRo2bnE4hgu13C/77xFlVFMaHskxV3yBtKFZvyDSWMw3eZ5FDkYcjcES9/gD6uCuy1BUifeTywyuLtj7oVRvicrSse8OVY26L1cVBWGY+VuHlJzdwC/NWZx2ZPvmi7v3P33e29mo96lIPWpz94qRE07qT9z7pisdG+lbxX/8NAQRpVcbknDJ8i35FdPDUK1Qxvn2JNVyf1OtgcPIohnFk39tc+S3kxuOjSg6HbQlGyz2d0lTLSX4j6zpXdkp6FEKet57SG1Ab2VmJjfCPNDeTGND3dnUyei/Kqv8t1jpDW+rcRjcq5GTQAjv8sR2XKx2iT+LDFWj4Frdkt0jMKe724VM6XHkIlLVfLa0niNPx0wIjysesfbU0Wtioa3ViiCFVr39mR5JhUj9ijWX/fw76+9IvCqjfBIGZGz0eLSV81C3zUcNzb+23fgy90zHz7hbyyZyLyLHwikxeR6L/h0y/uPtT3yHi3T2588x5kZV9fkAWIW0fh5POB6l7W3nRIykLfUHajVUdomjFU8hIZWUe5iOAQ7Omucwl3wxX6Dkn1HjVpz5HvHnoVD+OvKkBj3gK9aqlnlEV1LNtZ1+Mqfwf/Utcg5PQCNiCfneDYLntK/7baIe+sm3B0gsvY3Lun2x5XtXFOHeeutIklW0RlBMa4KMy2INr60VAZJ3Dwa1hA/bpPi7PsQ90bJc0tXRC6Q1bXkPqM20MCE1JAuXhUYFrGWvtLT6Wp/QE9YtH8rTacspGfuPWYCD7+frkUd76yj8WICBr350GyX1S8oRKN9bUubWKdDnY00jT5TmEJ1uby5BMIo+bS7Etisw9UwatGNS3kB9jWi8cp3Iun8ERPKTdS61AbBlPcD9CYkdwJxbJO7f3Nv06VtDSPXeYoUZ0HcHfxPDCpDM3r15yGkKZdnnfxUHzuV6lvoVhI96yFmo0nQLJFAQ4yYWJm3As6inApsTkap9FpnvMeXIbmLYpOsYEVBtc6DuXkgZ7Z11GT7LfOrQ8cWSNHu4Df63LG3ge1HIJFNgv5EbhGjPYHOlIFG9xvO47x3qb7LUy6W2cOKP2mArjkRswGICEnZASHbE9hLT9mQakQO1559eo9SKh6VU57dIc/NEvAUlO281LmuCnJUeX3C+aUvBGf8XvD8A6bhNJkppa7UalZHKBvkYp4jQyubwTIYfJbMrhNbyrLf3ICEkxWoyqI4fII5JrO86X6avKroeJEJRKkjLg3u4Yx7C+sqtb0sxUVCg1r6VImsLIeHjm9GTqF8UZTkVh4PZT6jLMxtxaQrObmbXaF+droNCpOf9oJN+sWxHZG8gXDE4JX+KVaXR4zDyzlrUp13fcHawEEDT+X6MFrJRNyvWiJuZc6XqnpcPZ0TBDi3v1Rp8M/Lst7kv2fOYVHmFwgba86nC21mTz8X6sS9GF+TXAKnEf/TIpHlIpAxbOT2PT+PofTinTqjiorJF4sVk0BTptZkZfQikbA1peQ9Qp02sUWlktydYE1zHLYiVvlQSK/mKZ0Y8kVB1Q8G6FrDp3OPz0dPmigHFo5DNi3xDoP3sLZRlUd6YNB2X9IGrdu2VT2gRuIwxP4rkPcKT2hAJYTUy7h3I8vOybiLkopb4eYeV4Oi7UAPaIrfWEWVgm8ho2GbCPTzJKIiFGqjcb9Ce6NEM4dvpmzrPbbQOd+tk2I0Zi7MT5LKkhzZwmd1ey6SyQzW4j5hbpRjm1ELs44teinJ1/uZkhITNGdLII6vGskHVUj+qdqV65fjXNEwE7+dJc/w/DNfm2g=="
)

COMP_CLASS_MAP = {"7-11k": 0, "7-11K": 0, "fiveN": 1, "freefried": 2, "pube": 3, "roblock": 4}

def decode_embedded_folds(expected_n: int = EXPECTED_FOLDS_N) -> np.ndarray:
    """Decode verified 31,546-row 5-fold SGKF assignment array."""
    raw = np.frombuffer(zlib.decompress(base64.b64decode(EMBEDDED_FOLDS_B64)), dtype=np.uint8)
    d0 = raw // 25
    d1 = (raw % 25) // 5
    d2 = raw % 5
    folds = np.empty(len(raw) * 3, dtype=np.int8)
    folds[0::3], folds[1::3], folds[2::3] = d0, d1, d2
    folds = folds[:expected_n].copy()
    assert len(folds) == expected_n
    digest = hashlib.sha256(folds.astype(np.uint8).tobytes()).hexdigest()[:16]
    assert digest == EXPECTED_FOLDS_SHA256, f"Embedded folds checksum mismatch: {digest} != {EXPECTED_FOLDS_SHA256}"
    return folds

def count_real_files(folder: str) -> int:
    """Count real JPGs while ignoring macOS ._ AppleDouble stub files and dotfiles."""
    if not os.path.isdir(folder):
        return 0
    return sum(1 for f in os.listdir(folder) if not f.startswith("."))

def is_data_ready(d: str) -> bool:
    return (
        os.path.isfile(os.path.join(d, "folds_sgkf.npy"))
        and os.path.isfile(os.path.join(d, "train.csv"))
        and os.path.isfile(os.path.join(d, "test.csv"))
        and count_real_files(os.path.join(d, "train")) >= 31546
        and count_real_files(os.path.join(d, "test")) >= 25889
    )

def sha256_file(path: str):
    h = hashlib.sha256()
    total = 0
    with open(path, "rb") as f:
        while True:
            buf = f.read(8 * 1024 * 1024)
            if not buf:
                break
            h.update(buf)
            total += len(buf)
    return total, h.hexdigest()

def _ensure_symlink_or_copy(src: str, dst: str) -> None:
    if os.path.islink(dst):
        if os.path.realpath(dst) == os.path.realpath(src) and os.path.isdir(dst):
            return
        os.unlink(dst)
    elif os.path.isdir(dst) and count_real_files(dst) > 0:
        return
    elif os.path.exists(dst):
        os.remove(dst)
    try:
        os.symlink(src, dst)
    except OSError:
        shutil.copytree(src, dst)

def _find_competition_data_dir(search_roots):
    """Locate directory containing train.csv, test.csv, train/ (>=31546), and test/ (>=25889)."""
    for root_base in search_roots:
        if not os.path.isdir(root_base):
            continue
        base_depth = root_base.rstrip(os.sep).count(os.sep)
        for cur, dirs, files in os.walk(root_base, followlinks=True):
            if cur.rstrip(os.sep).count(os.sep) - base_depth > 6:
                dirs[:] = []
                continue
            fset = set(files)
            dset = set(dirs)
            if "train.csv" in fset and "test.csv" in fset and "train" in dset and "test" in dset:
                if count_real_files(os.path.join(cur, "train")) >= 31546 and count_real_files(os.path.join(cur, "test")) >= 25889:
                    return cur
    return None

def prepare_competition_dir(src_dir: str, data_dir: str) -> None:
    """Normalize official Kaggle competition layout (0 GB image copy via symlinks + verified SGKF folds)."""
    os.makedirs(data_dir, exist_ok=True)
    _ensure_symlink_or_copy(os.path.join(src_dir, "train"), os.path.join(data_dir, "train"))
    _ensure_symlink_or_copy(os.path.join(src_dir, "test"), os.path.join(data_dir, "test"))

    tr_df = pd.read_csv(os.path.join(src_dir, "train.csv"))
    te_df = pd.read_csv(os.path.join(src_dir, "test.csv"))
    if "file_name" not in tr_df.columns:
        tr_df["file_name"] = tr_df["id"].astype(str) + ".jpg"
    if "label" not in tr_df.columns and "class" in tr_df.columns:
        tr_df["label"] = tr_df["class"].map(COMP_CLASS_MAP).astype(int)
    if "file_name" not in te_df.columns:
        te_df["file_name"] = te_df["id"].astype(str) + ".jpg"

    assert len(tr_df) == EXPECTED_FOLDS_N, f"Unexpected train row count: {len(tr_df)}"
    for fn in tr_df["file_name"].iloc[:3]:
        assert os.path.isfile(os.path.join(data_dir, "train", fn)), f"Missing train image: {fn}"
    for fn in te_df["file_name"].iloc[:3]:
        assert os.path.isfile(os.path.join(data_dir, "test", fn)), f"Missing test image: {fn}"

    tr_df.to_csv(os.path.join(data_dir, "train.csv"), index=False)
    te_df.to_csv(os.path.join(data_dir, "test.csv"), index=False)

    src_folds = os.path.join(src_dir, "folds_sgkf.npy")
    dst_folds = os.path.join(data_dir, "folds_sgkf.npy")
    if os.path.isfile(src_folds):
        shutil.copy2(src_folds, dst_folds)
    else:
        assert str(tr_df["id"].iloc[0]) in ("su7tkav", "PLY00001"), (
            f"Unexpected train.csv first id={tr_df['id'].iloc[0]} for embedded folds_sgkf"
        )
        np.save(dst_folds, decode_embedded_folds(len(tr_df)))

def prepare_dataset_pack(data_dir: str = DATA_DIR, search_roots=None) -> None:
    if is_data_ready(data_dir):
        print("Dataset already ready and verified in", data_dir)
        return
    os.makedirs(data_dir, exist_ok=True)
    if search_roots is None:
        search_roots = ["/kaggle/input", "/kaggle/working", "/content", "/content/drive/MyDrive", "."]

    # 1. Discover pack archive or split parts
    parts_dirs = []
    direct_candidates = []
    for rb in search_roots:
        if not os.path.isdir(rb):
            continue
        rb_depth = rb.rstrip(os.sep).count(os.sep)
        for cur, dirs, files in os.walk(rb, followlinks=True):
            if cur.rstrip(os.sep).count(os.sep) - rb_depth > 4:
                dirs[:] = []
                continue
            if os.path.basename(cur) == "task4_colab_pack_parts" and glob.glob(os.path.join(cur, "part_*")):
                parts_dirs.append(cur)
            for fn in ("task4_colab_pack.tar", "task4_colab_pack.zip"):
                if fn in files:
                    direct_candidates.append(os.path.join(cur, fn))

    merged_tar = os.path.join(os.path.dirname(data_dir), "task4_colab_pack_joined.tar")
    joined_from_parts = False
    parts_error = None
    pdir = parts_dirs[0] if parts_dirs else None
    if pdir and not any(os.path.isfile(c) for c in direct_candidates):
        expected_paths, bad_parts = [], []
        for idx in range(EXPECTED_NUM_PARTS):
            pname = f"part_{idx:02d}"
            exp_sz = EXPECTED_LAST_PART_BYTES if idx == EXPECTED_NUM_PARTS - 1 else EXPECTED_STD_PART_BYTES
            ppath = os.path.join(pdir, pname)
            if not os.path.isfile(ppath):
                bad_parts.append(f"{pname} (missing)")
            elif os.path.getsize(ppath) != exp_sz:
                bad_parts.append(f"{pname} ({os.path.getsize(ppath):,}B != {exp_sz:,}B)")
            else:
                expected_paths.append(ppath)
        if bad_parts:
            parts_error = f"Incomplete pack parts in {pdir}: {bad_parts[:8]}"
        else:
            print(f"Joining {len(expected_paths)} verified chunks from {pdir} -> {merged_tar} ...")
            h = hashlib.sha256()
            total = 0
            with open(merged_tar, "wb") as out_f:
                for pf in expected_paths:
                    with open(pf, "rb") as in_f:
                        while True:
                            buf = in_f.read(8 * 1024 * 1024)
                            if not buf:
                                break
                            out_f.write(buf)
                            h.update(buf)
                            total += len(buf)
            if total != EXPECTED_TAR_BYTES or h.hexdigest() != EXPECTED_TAR_SHA256:
                os.remove(merged_tar)
                raise RuntimeError(f"Joined archive checksum mismatch: bytes={total}, sha256={h.hexdigest()}")
            joined_from_parts = True
            direct_candidates.insert(0, merged_tar)
            print(f"Verified pack SHA-256: {EXPECTED_TAR_SHA256[:16]}... ({total / (1024**2):.1f} MiB)")

    pack_path = next((p for p in direct_candidates if os.path.isfile(p) and (p.endswith(".zip") or os.path.getsize(p) >= 700_000_000)), None)
    if pack_path is not None:
        print("Extracting", pack_path, "->", data_dir, "...")
        if pack_path.endswith(".zip"):
            with zipfile.ZipFile(pack_path, "r") as zf:
                zf.extractall(data_dir)
        else:
            subprocess.check_call(["tar", "--exclude=._*", "--exclude=*/._*", "-xf", pack_path, "-C", data_dir])
        if joined_from_parts and os.path.isfile(merged_tar):
            os.remove(merged_tar)
        return

    # 2. Zero-upload Kaggle competition layout resolver
    comp_dir = _find_competition_data_dir(search_roots)
    if comp_dir is not None:
        print(f"Detected Kaggle/Competition Task 4 dataset -> {comp_dir}")
        prepare_competition_dir(comp_dir, data_dir)
        print(f"Normalized symlinks + verified SGKF folds into {data_dir} (0 GB extra image disk used!)")
        return

    raise RuntimeError(
        parts_error
        or "Task 4 dataset not found! On Kaggle: click '+ Add Input' and add Competition 'player-intelligence-system-cpe-342-ml' (or task4_colab_pack)."
    )

if not os.environ.get("T4_SKIP_AUTO_PREP"):
    prepare_dataset_pack()
    assert is_data_ready(DATA_DIR), "Dataset setup incomplete!"
    print("Ready! Train JPGs:", count_real_files(f"{DATA_DIR}/train"), "| Test JPGs:", count_real_files(f"{DATA_DIR}/test"))

In [ ]:
# Cell 3: Fast Spatial + GPU-Vectorized Nightmare Augmentations & ExtHard Validator
import io
import numpy as np
import torch
from PIL import Image, ImageFilter

def apply_nightmare_spatial_u8(pil_img: Image.Image, rng: np.random.Generator) -> np.ndarray:
    """Fast CPU spatial/blur/JPEG nightmare transform returning uint8 (photometric jitter runs on GPU)."""
    if rng.random() < 0.50:
        pil_img = pil_img.transpose(Image.FLIP_LEFT_RIGHT)
    if rng.random() < 0.55:
        ang = float(rng.uniform(-35.0, 35.0))
        pil_img = pil_img.rotate(ang, resample=Image.BILINEAR, fillcolor=(0, 0, 0))
    if rng.random() < 0.16:
        pil_img = pil_img.filter(ImageFilter.GaussianBlur(radius=float(rng.uniform(0.4, 1.3))))
    if rng.random() < 0.20:
        buf = io.BytesIO()
        pil_img.save(buf, format="JPEG", quality=int(rng.integers(28, 75)))
        buf.seek(0)
        pil_img = Image.open(buf).convert("RGB")
    return np.asarray(pil_img, dtype=np.uint8)

def apply_nightmare_photo_torch(x: torch.Tensor) -> torch.Tensor:
    """Vectorized GPU photometric nightmare corruptions on (N, 3, H, W) float32 in [0, 1]."""
    n = x.shape[0]
    if n == 0:
        return x
    dev = x.device
    # 1. Neon channel gain / color tint (p=0.55)
    m_tint = (torch.rand(n, device=dev) < 0.55).view(n, 1, 1, 1)
    gains = torch.empty((n, 3, 1, 1), device=dev, dtype=torch.float32).uniform_(0.30, 2.20)
    gains = gains / gains.mean(dim=1, keepdim=True).clamp(min=1e-4)
    x = torch.where(m_tint, (x * gains).clamp(0.0, 1.0), x)

    # 2. Exposure & contrast corruption (0.20 dark gamma / 0.20 overexpose / 0.14 washed)
    r_exp = torch.rand(n, device=dev)
    m_dark = (r_exp < 0.20).view(n, 1, 1, 1)
    m_over = ((r_exp >= 0.20) & (r_exp < 0.40)).view(n, 1, 1, 1)
    m_wash = ((r_exp >= 0.40) & (r_exp < 0.54)).view(n, 1, 1, 1)

    gamma = torch.empty((n, 1, 1, 1), device=dev).uniform_(1.8, 3.0)
    dark_sc = torch.empty((n, 1, 1, 1), device=dev).uniform_(0.32, 0.70)
    x_dark = x.clamp(min=1e-4).pow(gamma) * dark_sc

    ov_sc = torch.empty((n, 1, 1, 1), device=dev).uniform_(1.25, 2.10)
    ov_sh = torch.empty((n, 1, 1, 1), device=dev).uniform_(0.24, 0.48)
    x_over = (x * ov_sc + ov_sh).clamp(0.0, 1.0)

    w_sc = torch.empty((n, 1, 1, 1), device=dev).uniform_(0.30, 0.65)
    w_sh = torch.empty((n, 1, 1, 1), device=dev).uniform_(-0.15, 0.15)
    x_wash = (0.5 + (x - 0.5) * w_sc + w_sh).clamp(0.0, 1.0)

    x = torch.where(m_dark, x_dark, torch.where(m_over, x_over, torch.where(m_wash, x_wash, x)))

    # 3. Gaussian sensor noise (p=0.40)
    m_noise = (torch.rand(n, device=dev) < 0.40).view(n, 1, 1, 1)
    sig = torch.empty((n, 1, 1, 1), device=dev).uniform_(0.015, 0.075)
    x = torch.where(m_noise, (x + torch.randn_like(x) * sig).clamp(0.0, 1.0), x)
    return x

def apply_nightmare_aug(pil_img: Image.Image, rng: np.random.Generator) -> np.ndarray:
    """CPU nightmare helper retained for deterministic parity checks."""
    if rng.random() < 0.50:
        pil_img = pil_img.transpose(Image.FLIP_LEFT_RIGHT)
    if rng.random() < 0.55:
        ang = float(rng.uniform(-35.0, 35.0))
        pil_img = pil_img.rotate(ang, resample=Image.BILINEAR, fillcolor=(0, 0, 0))
    if rng.random() < 0.16:
        pil_img = pil_img.filter(ImageFilter.GaussianBlur(radius=float(rng.uniform(0.4, 1.3))))
    arr = np.asarray(pil_img, dtype=np.float32) * (1.0 / 255.0)
    if rng.random() < 0.55:
        gains = rng.uniform(0.30, 2.20, size=(1, 1, 3)).astype(np.float32)
        gains /= float(gains.mean())
        arr = np.clip(arr * gains, 0.0, 1.0)
    r_exp = rng.random()
    if r_exp < 0.20:
        arr = np.power(np.clip(arr, 1e-4, 1.0), float(rng.uniform(1.8, 3.0))) * float(rng.uniform(0.32, 0.70))
    elif r_exp < 0.40:
        arr = np.clip(arr * float(rng.uniform(1.25, 2.10)) + float(rng.uniform(0.24, 0.48)), 0.0, 1.0)
    elif r_exp < 0.54:
        arr = np.clip(0.5 + (arr - 0.5) * float(rng.uniform(0.30, 0.65)) + float(rng.uniform(-0.15, 0.15)), 0.0, 1.0)
    if rng.random() < 0.40:
        arr = arr + rng.normal(0.0, float(rng.uniform(0.015, 0.075)), size=arr.shape).astype(np.float32)
    arr_u8 = np.clip(arr * 255.0, 0.0, 255.0).astype(np.uint8)
    if rng.random() < 0.20:
        buf = io.BytesIO()
        Image.fromarray(arr_u8).save(buf, format="JPEG", quality=int(rng.integers(28, 75)))
        buf.seek(0)
        arr_u8 = np.asarray(Image.open(buf).convert("RGB"), dtype=np.uint8)
    return arr_u8

def make_deterministic_exthard(jpeg_bytes_list, indices: np.ndarray, seed: int = 123):
    """Generate deterministic ExtremeHard validation images for clean fold-level stress testing."""
    rng = np.random.default_rng(seed)
    out = []
    for k, idx in enumerate(indices):
        im = Image.open(io.BytesIO(jpeg_bytes_list[int(idx)])).convert("RGB")
        im = im.rotate(float(rng.uniform(-35.0, 35.0)), resample=Image.BILINEAR, fillcolor=(0, 0, 0))
        arr = np.asarray(im, dtype=np.float32) * (1.0 / 255.0)
        mode = k % 4
        if mode == 0:
            arr = np.power(np.clip(arr, 1e-4, 1.0), float(rng.uniform(1.8, 2.8))) * float(rng.uniform(0.35, 0.65))
        elif mode == 1:
            arr = np.clip(arr * float(rng.uniform(1.3, 2.0)) + float(rng.uniform(0.30, 0.50)), 0.0, 1.0)
        elif mode == 2:
            g = rng.uniform(0.35, 2.0, size=(1, 1, 3)).astype(np.float32)
            g /= float(g.max())
            arr = np.clip(np.power(arr, 0.85) * g * float(rng.uniform(0.8, 1.4)), 0.0, 1.0)
        else:
            arr = np.clip(0.5 + (arr - 0.5) * float(rng.uniform(0.4, 0.7)) + rng.uniform(-0.15, 0.15, (1, 1, 3)), 0.0, 1.0)
        arr = arr + rng.normal(0.0, float(rng.uniform(0.03, 0.07)), size=arr.shape).astype(np.float32)
        out.append(np.clip(arr * 255.0, 0.0, 255.0).astype(np.uint8))
    return out

print("Fast spatial + GPU-vectorized Nightmare & ExtHard helpers ready.")

In [ ]:
# Cell 4: Interior Gray-World Normalization (CPU Train View + Vectorized GPU Eval View)
import math
import numpy as np
import torch

def interior_gw_stretch_np(arr_u8: np.ndarray) -> np.ndarray:
    """Fast Gray-World + Percentile Contrast Stretch using central 60% interior pixels (CPU)."""
    h, w, _ = arr_u8.shape
    h0, h1 = int(h * 0.2), int(h * 0.8)
    w0, w1 = int(w * 0.2), int(w * 0.8)
    b = arr_u8.astype(np.float32) * (1.0 / 255.0)
    patch = b[h0:h1:2, w0:w1:2, :]
    ch_m = np.clip(patch.mean(axis=(0, 1), keepdims=True), 0.025, 1.0)
    b_gw = np.clip(b * (ch_m.mean(axis=2, keepdims=True) / ch_m), 0.0, 1.0)
    p_sub = b_gw[h0:h1:3, w0:w1:3, :].ravel()
    lo, hi = np.quantile(p_sub, (0.02, 0.98))
    b_st = np.clip((b_gw - float(lo)) / max(float(hi - lo), 0.10), 0.0, 1.0)
    m_int = float(np.clip(b_st[h0:h1:3, w0:w1:3, :].mean(), 0.05, 0.95))
    gamma = float(np.clip(math.log(0.45) / math.log(m_int), 0.50, 1.80))
    return np.clip(np.power(b_st, gamma), 0.0, 1.0).astype(np.float32)

def interior_gw_stretch_torch(b: torch.Tensor) -> torch.Tensor:
    """Vectorized GPU Gray-World + Percentile Contrast Stretch matching interior_gw_stretch_np."""
    _, _, h, w = b.shape
    h0, h1 = int(h * 0.2), int(h * 0.8)
    w0, w1 = int(w * 0.2), int(w * 0.8)
    patch = b[:, :, h0:h1:2, w0:w1:2]
    ch_m = patch.mean(dim=(2, 3), keepdim=True).clamp(0.025, 1.0)
    b_gw = (b * (ch_m.mean(dim=1, keepdim=True) / ch_m)).clamp(0.0, 1.0)
    p_sub = b_gw[:, :, h0:h1:3, w0:w1:3].reshape(b.shape[0], -1)
    q = torch.tensor([0.02, 0.98], device=b.device, dtype=b.dtype)
    qs = torch.quantile(p_sub, q, dim=1)
    lo = qs[0].view(-1, 1, 1, 1)
    hi = qs[1].view(-1, 1, 1, 1)
    b_st = ((b_gw - lo) / (hi - lo).clamp(min=0.10)).clamp(0.0, 1.0)
    m_int = b_st[:, :, h0:h1:3, w0:w1:3].mean(dim=(1, 2, 3)).clamp(0.05, 0.95)
    gamma = (math.log(0.45) / torch.log(m_int)).clamp(0.50, 1.80).view(-1, 1, 1, 1)
    return b_st.pow(gamma).clamp(0.0, 1.0)

print("CPU + Vectorized GPU Interior Gray-World normalizers ready.")

In [ ]:
# Cell 5: JPEG RAM Preloader & Zero-Overhead uint8 Train/Eval Datasets (All Photometrics/Rotations Run on GPU)
import io, os
import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset

def load_jpeg_bytes(folder: str, filenames):
    """Preload compressed JPEG byte buffers into host RAM to eliminate disk I/O stalls."""
    buffers = []
    for fn in filenames:
        with open(os.path.join(folder, fn), "rb") as f:
            buffers.append(f.read())
    return buffers

class Task4TrainDataset(Dataset):
    """Fast worker dataset returning uint8 (3, H, W) + is_nightmare flag; GPU handles photometrics & GW norm."""

    def __init__(self, jpeg_bytes, indices, labels, img_hw, p_nightmare=0.50, p_norm_view=0.25, seed=42):
        self.jpeg_bytes = jpeg_bytes
        self.indices = indices
        self.labels = labels
        self.img_hw = img_hw
        self.p_nightmare = p_nightmare
        self.p_norm_view = p_norm_view
        self.seed = seed
        self._rng = None

    def _get_rng(self):
        if self._rng is None:
            info = torch.utils.data.get_worker_info()
            wid = info.id if info is not None else 0
            self._rng = np.random.default_rng(self.seed + wid * 7919)
        return self._rng

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, pos: int):
        rng = self._get_rng()
        idx = int(self.indices[pos])
        pil = Image.open(io.BytesIO(self.jpeg_bytes[idx])).convert("RGB")
        target_wh = (self.img_hw[1], self.img_hw[0])
        if pil.size != target_wh:
            pil = pil.resize(target_wh, resample=Image.BILINEAR)

        is_nm = bool(rng.random() < self.p_nightmare)
        if is_nm:
            arr_u8 = apply_nightmare_spatial_u8(pil, rng)
        else:
            if rng.random() < 0.50:
                pil = pil.transpose(Image.FLIP_LEFT_RIGHT)
            arr_u8 = np.asarray(pil, dtype=np.uint8)

        t_u8 = torch.from_numpy(np.array(arr_u8, dtype=np.uint8, copy=True)).permute(2, 0, 1).contiguous()
        return t_u8, is_nm, torch.tensor(int(self.labels[idx]), dtype=torch.long)

class Task4RawViewsDataset(Dataset):
    """Single-decode 1-frame uint8 dataset; flip, +-18deg rotation & Interior-GW run vectorized on GPU."""

    def __init__(self, source_items, indices, img_hw, need_rot: bool = False, is_bytes: bool = True):
        self.source_items = source_items
        self.indices = indices if indices is not None else np.arange(len(source_items))
        self.img_hw = img_hw
        self.need_rot = False  # Always perform +-18deg rotation on GPU via grid_sample
        self.is_bytes = is_bytes

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, pos: int):
        item = self.source_items[int(self.indices[pos])]
        pil = Image.open(io.BytesIO(item)).convert("RGB") if self.is_bytes else Image.fromarray(item)
        target_wh = (self.img_hw[1], self.img_hw[0])
        if pil.size != target_wh:
            pil = pil.resize(target_wh, resample=Image.BILINEAR)
        arr_u8 = np.array(pil, dtype=np.uint8, copy=True)
        return torch.from_numpy(arr_u8).permute(2, 0, 1).contiguous()

print("Zero-overhead uint8 train/eval datasets ready.")

In [ ]:
# Cell 6: Model Specs (V3 + SigLIP Gen-2 + H100 Turbo), Smart Freeze, VRAM Batch/GC Selector & Optimizer
import math
from dataclasses import dataclass
from typing import Tuple
import timm
import torch
import torch.nn as nn

@dataclass(frozen=True)
class ModelSpec:
    """Configuration for a corruption-hardened Task 4 vision backbone."""
    key: str
    timm_name: str
    img_hw: Tuple[int, int]
    epochs: int
    base_lr: float
    backbone_lr_mult: float
    batch_h100: int
    eval_batch_h100: int
    batch_a100: int
    batch_l4: int
    batch_t4: int
    eval_batch_t4: int
    grad_ckpt: bool
    use_img_size_arg: bool
    channels_last_ampere: bool
    default_freeze_vit_blocks: int

MODEL_SPECS = {
    "cnxv3": ModelSpec("cnxv3", "convnext_base.dinov3_lvd1689m", (192, 384), 6, 3.0e-4, 0.25, 96, 256, 96, 64, 48, 160, False, False, True, 0),
    "cnxv2b": ModelSpec("cnxv2b", "convnextv2_base.fcmae_ft_in22k_in1k", (192, 384), 6, 3.0e-4, 0.25, 96, 256, 96, 64, 48, 160, False, False, True, 0),
    "eva02l": ModelSpec("eva02l", "eva02_large_patch14_448.mim_m38m_ft_in22k_in1k", (196, 392), 4, 2.0e-4, 0.22, 48, 192, 48, 32, 16, 96, True, True, False, 6),
    "dinov3l": ModelSpec("dinov3l", "vit_large_patch16_dinov3.lvd1689m", (192, 384), 5, 2.0e-4, 0.22, 48, 192, 64, 32, 16, 128, True, True, False, 6),
    "siglip_so400m": ModelSpec("siglip_so400m", "vit_so400m_patch14_siglip_384", (196, 392), 4, 1.6e-4, 0.18, 48, 160, 48, 24, 12, 72, True, True, False, 10),
    "siglip_l16": ModelSpec("siglip_l16", "vit_large_patch16_siglip_384", (192, 384), 5, 2.0e-4, 0.20, 48, 192, 64, 32, 16, 96, True, True, False, 8),
}

def apply_t4_freeze(model: nn.Module, spec: ModelSpec, freeze_vit_blocks: int):
    """Freeze only lowest ViT blocks (never patch_embed/tokens/head/attn_pool or ConvNeXt stem)."""
    total_params = sum(p.numel() for p in model.parameters())
    if spec.default_freeze_vit_blocks > 0 and freeze_vit_blocks > 0:
        for name, param in model.named_parameters():
            if name.startswith("blocks."):
                parts = name.split(".")
                if len(parts) > 1 and parts[1].isdigit() and int(parts[1]) < freeze_vit_blocks:
                    param.requires_grad_(False)
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return trainable_params, total_params

def build_model(spec: ModelSpec, pretrained: bool = True, freeze_vit_blocks=None, disable_grad_ckpt: bool = False):
    """Instantiate timm model with exact 2:1 aspect-ratio img_size, smart freeze, and RGB mean/std."""
    fb = spec.default_freeze_vit_blocks if freeze_vit_blocks is None else int(freeze_vit_blocks)
    kwargs = {"pretrained": pretrained, "num_classes": 5}
    if spec.use_img_size_arg:
        kwargs["img_size"] = spec.img_hw
    model = timm.create_model(spec.timm_name, **kwargs)
    if spec.grad_ckpt and not disable_grad_ckpt and hasattr(model, "set_grad_checkpointing"):
        model.set_grad_checkpointing(True)
    param_counts = apply_t4_freeze(model, spec, freeze_vit_blocks=fb)
    cfg = timm.data.resolve_model_data_config(model)
    return model, tuple(cfg["mean"]), tuple(cfg["std"]), param_counts

def pick_batch_and_accum(spec: ModelSpec, device: torch.device, target_eff_bs: int = 48, disable_grad_ckpt_high_vram: bool = True):
    """Select train batch size, grad accumulation, eval batch size, and H100 grad-ckpt bypass."""
    if device.type != "cuda":
        return 8, 1, 16, False
    dev_idx = 0 if device.index is None else int(device.index)
    vram_gb = torch.cuda.get_device_properties(dev_idx).total_memory / (1024**3)
    disable_gc = False
    if vram_gb >= 60.0:
        bs, eval_bs = spec.batch_h100, spec.eval_batch_h100
        disable_gc = bool(disable_grad_ckpt_high_vram)
    elif vram_gb >= 36.0:
        bs, eval_bs = spec.batch_a100, 128
    elif vram_gb >= 21.0:
        bs, eval_bs = spec.batch_l4, 96
    else:
        bs, eval_bs = spec.batch_t4, spec.eval_batch_t4
    return bs, max(1, math.ceil(target_eff_bs / bs)), eval_bs, disable_gc

def make_param_groups(model: nn.Module, base_lr: float, bb_mult: float, wd: float = 2e-2):
    """Separate classifier/attn_pool heads from backbone weights and exclude 1D norm/bias from decay."""
    hd_d, hd_n, bb_d, bb_n = [], [], [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        is_h = name.startswith("head") or name.startswith("fc") or name.startswith("attn_pool") or ".head." in name
        no_w = p.ndim <= 1 or name.endswith(".bias") or "norm" in name.lower()
        target = (hd_n if no_w else hd_d) if is_h else (bb_n if no_w else bb_d)
        target.append(p)
    groups = []
    for lst, lr_v, wd_v in [(hd_d, base_lr, wd), (hd_n, base_lr, 0.0), (bb_d, base_lr * bb_mult, wd), (bb_n, base_lr * bb_mult, 0.0)]:
        if lst:
            groups.append({"params": lst, "lr": lr_v, "weight_decay": wd_v})
    return groups

def update_ema(ema_state, model: nn.Module, decay: float = 0.996):
    """Update exponential moving average shadow weights on device."""
    with torch.no_grad():
        for k, v in model.state_dict().items():
            if k in ema_state and v.dtype.is_floating_point:
                ema_state[k].mul_(decay).add_(v.detach(), alpha=1.0 - decay)
            elif k not in ema_state:
                ema_state[k] = v.detach().clone()

print("Model specs & optimizer helpers ready:", list(MODEL_SPECS.keys()))

In [ ]:
# Cell 7: CUDA Runtime Engine Preflight Probe (Supports Multi-GPU Index & cuDNN-Safe Fallback)
import torch
import torch.nn as nn
import torch.nn.functional as F

def probe_runtime_engine(model: nn.Module, spec: ModelSpec, device: torch.device, amp_dtype, use_scaler: bool):
    """Test forward+backward and odd-batch eval before Epoch 0; falls back cleanly if cuDNN FIND fails."""
    h, w = spec.img_hw
    if device.type != "cuda":
        xb = torch.randn(2, 3, h, w, device=device)
        yb = torch.tensor([0, 1], device=device, dtype=torch.long)
        model.train()
        loss = F.cross_entropy(model(xb).float(), yb)
        loss.backward()
        assert any(p.grad is not None for p in model.parameters() if p.requires_grad), "No trainable grads after backward"
        model.zero_grad(set_to_none=True)
        model.eval()
        with torch.no_grad():
            _ = model(xb)
        return False, f"{device.type}-fp32"

    dev_idx = 0 if device.index is None else int(device.index)
    torch.cuda.set_device(device)
    cc_major, cc_minor = torch.cuda.get_device_capability(dev_idx)
    candidates = []
    if cc_major >= 8:
        if spec.channels_last_ampere:
            candidates.append((True, True, True, f"sm{cc_major}{cc_minor}-nhwc-cudnn"))
        candidates.append((False, True, True, f"sm{cc_major}{cc_minor}-nchw-cudnn"))
        candidates.append((False, True, False, f"sm{cc_major}{cc_minor}-nchw-cudnn-safe"))
        candidates.append((False, False, False, f"sm{cc_major}{cc_minor}-nchw-nocudnn"))
    else:
        candidates.append((False, True, False, f"sm{cc_major}{cc_minor}-nchw-cudnn-safe"))
        candidates.append((False, False, False, f"sm{cc_major}{cc_minor}-nchw-nocudnn"))

    last_exc = None
    with torch.cuda.device(device):
        for use_cl, use_cudnn, use_bench, tag in candidates:
            try:
                torch.backends.cudnn.enabled = use_cudnn
                torch.backends.cudnn.benchmark = use_bench
                model.to(memory_format=(torch.channels_last if use_cl else torch.contiguous_format))
                model.zero_grad(set_to_none=True)

                probe_scaler = torch.amp.GradScaler("cuda", init_scale=1024.0, enabled=use_scaler)
                xb = torch.randn(2, 3, h, w, device=device)
                if use_cl:
                    xb = xb.to(memory_format=torch.channels_last)
                yb = torch.tensor([0, 1], device=device, dtype=torch.long)

                model.train()
                with torch.amp.autocast(device.type, dtype=amp_dtype) if amp_dtype is not None else torch.enable_grad():
                    logits = model(xb)
                    assert torch.isfinite(logits.float()).all(), "Non-finite logits in preflight probe"
                    loss = F.cross_entropy(logits.float(), yb)
                if use_scaler:
                    probe_scaler.scale(loss).backward()
                else:
                    loss.backward()
                has_grad = any(p.grad is not None for p in model.parameters() if p.requires_grad)
                assert has_grad, "Gradients missing in preflight probe"
                model.zero_grad(set_to_none=True)

                model.eval()
                xb_odd = torch.randn(3, 3, h, w, device=device)
                if use_cl:
                    xb_odd = xb_odd.to(memory_format=torch.channels_last)
                with torch.no_grad(), (torch.amp.autocast(device.type, dtype=amp_dtype) if amp_dtype is not None else torch.no_grad()):
                    _ = model(xb_odd)
                torch.cuda.synchronize(device)
                return use_cl, tag
            except Exception as exc:
                last_exc = exc
                model.zero_grad(set_to_none=True)
                torch.cuda.empty_cache()

    raise RuntimeError(f"All runtime engine candidates failed for {spec.key}: {last_exc}") from last_exc

print("Runtime engine preflight probe ready.")

In [ ]:
# Cell 8: Vectorized GPU 6-View TTA Inference, Geometric Pooling & Thread/Atomic Epoch+Fold Checkpointers
import math, os, shutil, threading
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

def gpu_rotate_batch(x: torch.Tensor, angle_deg: float) -> torch.Tensor:
    """Aspect-preserving GPU bilinear rotation matching PIL.Image.rotate(angle_deg, BILINEAR)."""
    b, _, h, w = x.shape
    theta = math.radians(angle_deg)
    cos_t, sin_t = math.cos(theta), math.sin(theta)
    mat = torch.tensor(
        [[cos_t, -(h / w) * sin_t, 0.0],
         [(w / h) * sin_t, cos_t, 0.0]],
        device=x.device, dtype=x.dtype,
    ).unsqueeze(0).expand(b, -1, -1)
    grid = F.affine_grid(mat, size=x.size(), align_corners=False)
    return F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)

def _build_gpu_tta_batch(raw_f: torch.Tensor, views):
    """Expand raw (B, 3, H, W) [or (B, R, 3, H, W)] float32 [0,1] frames into (B, V, 3, H, W) on GPU."""
    x_orig = raw_f[:, 0] if raw_f.ndim == 5 else raw_f
    x_flip = torch.flip(x_orig, dims=[-1]) if any(v in ("flip", "gw_flip") for v in views) else None
    x_gw = interior_gw_stretch_torch(x_orig) if "gw" in views else None
    x_gw_flip = interior_gw_stretch_torch(x_flip) if "gw_flip" in views else None
    built = []
    for v in views:
        if v == "orig":
            built.append(x_orig)
        elif v == "flip":
            built.append(x_flip)
        elif v == "gw":
            built.append(x_gw)
        elif v == "gw_flip":
            built.append(x_gw_flip)
        elif v == "rot_p18_gw":
            x_r = raw_f[:, 1] if (raw_f.ndim == 5 and raw_f.shape[1] >= 2) else gpu_rotate_batch(x_orig, 18.0)
            built.append(interior_gw_stretch_torch(x_r))
        elif v == "rot_m18_gw":
            x_r = raw_f[:, 2] if (raw_f.ndim == 5 and raw_f.shape[1] >= 3) else gpu_rotate_batch(x_orig, -18.0)
            built.append(interior_gw_stretch_torch(x_r))
        else:
            raise ValueError(f"Unknown TTA view: {v}")
    return torch.stack(built, dim=1)

@torch.no_grad()
def predict_views(
    model, source_items, indices, spec: ModelSpec, mean_t, std_t, views, eval_bs: int, device, amp_dtype,
    use_channels_last=False, is_bytes=True, num_workers=2, pin_memory=None,
):
    """Run 1-frame uint8 decode + GPU rotation/GW 6-view TTA with grad-checkpointing disabled and OOM backoff."""
    model.eval()
    was_gc = bool(getattr(model, "grad_checkpointing", False))
    if was_gc and hasattr(model, "set_grad_checkpointing"):
        model.set_grad_checkpointing(False)

    n = len(indices) if indices is not None else len(source_items)
    v_cnt = len(views)
    dl_bs = max(64, min(192, eval_bs)) if v_cnt == 1 else max(32, min(96, eval_bs))
    eff_pin = (device.type == "cuda") if pin_memory is None else bool(pin_memory)
    ds = Task4RawViewsDataset(source_items, indices, spec.img_hw, need_rot=False, is_bytes=is_bytes)
    dl_kw = {"batch_size": dl_bs, "shuffle": False, "num_workers": num_workers, "pin_memory": eff_pin}
    if num_workers > 0:
        dl_kw["prefetch_factor"] = 4
    dl = DataLoader(ds, **dl_kw)

    cur_bs = int(eval_bs)
    all_probs = []
    try:
        for batch_u8 in dl:
            bsz = batch_u8.shape[0]
            raw_f = batch_u8.to(device, non_blocking=eff_pin).float() * (1.0 / 255.0)
            views_t = _build_gpu_tta_batch(raw_f, views)
            flat = views_t.view(bsz * v_cnt, 3, views_t.shape[3], views_t.shape[4])
            out_chunks = []
            s = 0
            while s < len(flat):
                xb = flat[s : s + cur_bs]
                try:
                    if use_channels_last and device.type == "cuda":
                        xb = xb.to(memory_format=torch.channels_last)
                    xb = (xb - mean_t) / std_t
                    with torch.amp.autocast(device.type, dtype=amp_dtype) if amp_dtype is not None else torch.no_grad():
                        logits = model(xb)
                    out_chunks.append(F.softmax(logits.float(), dim=1).cpu().numpy())
                    s += len(xb)
                except RuntimeError as exc:
                    if "out of memory" not in str(exc).lower():
                        raise
                    if device.type == "cuda":
                        torch.cuda.empty_cache()
                    if cur_bs <= 16:
                        raise
                    cur_bs = max(16, cur_bs // 2)
                    print(f"[{spec.key}] Eval OOM backoff -> eval_bs={cur_bs}", flush=True)
            all_probs.append(np.concatenate(out_chunks, axis=0).reshape(bsz, v_cnt, 5))
    finally:
        if was_gc and hasattr(model, "set_grad_checkpointing"):
            model.set_grad_checkpointing(True)

    merged = np.concatenate(all_probs, axis=0) if all_probs else np.zeros((0, v_cnt, 5), dtype=np.float32)
    assert len(merged) == n
    return {v: merged[:, i, :].copy() for i, v in enumerate(views)}

def combine_tta_views(vdict, mode: str) -> np.ndarray:
    """Combine view probabilities in geometric log-space."""
    if mode == "orig2":
        keys, ws = ["orig", "flip"], [0.55, 0.45]
    elif mode == "weighted6":
        keys, ws = ["orig", "flip", "gw", "gw_flip", "rot_p18_gw", "rot_m18_gw"], [0.28, 0.20, 0.20, 0.12, 0.10, 0.10]
    else:
        raise ValueError(f"Unknown combine mode: {mode}")
    lp = sum(w * np.log(np.clip(vdict[k], 1e-7, 1.0)) for k, w in zip(keys, ws))
    lp -= lp.max(axis=1, keepdims=True)
    ep = np.exp(lp)
    return (ep / ep.sum(axis=1, keepdims=True)).astype(np.float32)

def save_csv_preds(out_path: str, df_meta: pd.DataFrame, probs: np.ndarray, folds_col=None):
    """Write predictions matching project/task4/predictions/model-level CSV schema."""
    fnames = df_meta["file_name"] if "file_name" in df_meta.columns else (df_meta["id"].astype(str) + ".jpg")
    out = pd.DataFrame({"id": df_meta["id"].values, "file_name": fnames.values})
    if "label" in df_meta.columns and df_meta["label"].notna().all():
        out["label"] = df_meta["label"].astype(int).values
    if folds_col is not None:
        out["fold"] = folds_col.astype(int)
    for c in range(5):
        out[f"prob_{c}"] = np.round(probs[:, c].astype(np.float64), 7)
    out.to_csv(out_path, index=False)

def atomic_savez_compressed(path: str, **arrays):
    """Atomically write compressed .npz file so disconnects never corrupt checkpoints."""
    tmp_path = f"{path}.tmp.{os.getpid()}.{threading.get_ident()}.npz"
    np.savez_compressed(tmp_path, **arrays)
    os.replace(tmp_path, path)

def load_fold_npz(path: str):
    """Safely load completed fold checkpoint if present and intact."""
    req = ("va_orig2", "va_w6", "ext_w6", "te_orig2", "te_w6")
    if not os.path.isfile(path):
        return None
    try:
        with np.load(path) as z:
            if not all(k in z for k in req):
                return None
            return {k: z[k].copy() for k in req}
    except Exception:
        return None

def resolve_epoch_ckpt_paths(cache_dir: str, model_key: str, k: int):
    primary_ckpt = os.path.join(cache_dir, f"fold{k}_ckpt.pt")
    if os.path.isdir("/kaggle/working"):
        base_tmp = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp"
        local_dir = os.path.join(base_tmp, "t4_hard_ckpt_tmp", f"folds_{model_key}")
        os.makedirs(local_dir, exist_ok=True)
        local_ckpt = os.path.join(local_dir, f"fold{k}_ckpt.pt")
    elif os.path.isdir("/content"):
        local_dir = os.path.join("/content/t4_hard_ckpt_tmp", f"folds_{model_key}")
        os.makedirs(local_dir, exist_ok=True)
        local_ckpt = os.path.join(local_dir, f"fold{k}_ckpt.pt")
    else:
        local_ckpt = primary_ckpt
    return primary_ckpt, local_ckpt

def save_epoch_ckpt(cache_dir: str, spec: ModelSpec, k: int, ep: int, epochs: int, freeze_vit_blocks: int, model, ema_state, opt, sched, scaler) -> str:
    """Save only trainable/buffer tensors + optimizer/scheduler state atomically; keeps Kaggle working quota safe."""
    primary_ckpt, local_ckpt = resolve_epoch_ckpt_paths(cache_dir, spec.key, k)
    trainable_keys = {n for n, p in model.named_parameters() if p.requires_grad}
    buf_keys = {n for n, _ in model.named_buffers()}
    keep_keys = trainable_keys | buf_keys

    payload = {
        "spec_key": spec.key,
        "img_hw": tuple(spec.img_hw),
        "epochs": int(epochs),
        "freeze_vit_blocks": int(freeze_vit_blocks),
        "epoch": int(ep),
        "trainable_model": {sk: sv.detach().cpu() for sk, sv in model.state_dict().items() if sk in keep_keys},
        "ema_state": {ek: ev.detach().cpu() for ek, ev in ema_state.items() if ek in keep_keys},
        "opt": opt.state_dict(),
        "sched": sched.state_dict(),
        "scaler": scaler.state_dict(),
    }

    target = primary_ckpt
    try:
        min_free = 10_000_000_000 if os.path.isdir("/kaggle/working") else 3_200_000_000
        if primary_ckpt != local_ckpt and shutil.disk_usage(cache_dir).free < min_free:
            target = local_ckpt
    except OSError:
        target = local_ckpt

    tmp_path = f"{target}.tmp.{os.getpid()}.{threading.get_ident()}"
    try:
        torch.save(payload, tmp_path)
        os.replace(tmp_path, target)
    except OSError:
        if os.path.isfile(tmp_path):
            os.remove(tmp_path)
        if target != local_ckpt:
            tmp_fb = f"{local_ckpt}.tmp.{os.getpid()}.{threading.get_ident()}"
            torch.save(payload, tmp_fb)
            os.replace(tmp_fb, local_ckpt)
            target = local_ckpt
        else:
            raise
    return target

def load_epoch_ckpt(cache_dir: str, spec: ModelSpec, k: int, epochs: int, freeze_vit_blocks: int):
    primary_ckpt, local_ckpt = resolve_epoch_ckpt_paths(cache_dir, spec.key, k)
    cands = [primary_ckpt] if primary_ckpt == local_ckpt else [primary_ckpt, local_ckpt]
    best_payload = None
    for cp in cands:
        if not os.path.isfile(cp):
            continue
        try:
            pl = torch.load(cp, map_location="cpu", weights_only=False)
            if (
                pl.get("spec_key") == spec.key
                and tuple(pl.get("img_hw", ())) == tuple(spec.img_hw)
                and int(pl.get("epochs", -1)) == int(epochs)
                and int(pl.get("freeze_vit_blocks", -1)) == int(freeze_vit_blocks)
            ):
                if best_payload is None or int(pl.get("epoch", -1)) > int(best_payload.get("epoch", -1)):
                    best_payload = pl
        except Exception:
            continue
    return best_payload

def remove_epoch_ckpt(cache_dir: str, model_key: str, k: int):
    primary_ckpt, local_ckpt = resolve_epoch_ckpt_paths(cache_dir, model_key, k)
    for cp in {primary_ckpt, local_ckpt}:
        if os.path.isfile(cp):
            os.remove(cp)

print("GPU TTA inference & epoch/fold atomic checkpoint helpers ready.")

In [ ]:
# Cell 9: Single-Fold Trainer (H100 Turbo / Multi-GPU Safe / Epoch Auto-Resume + EMA + 6-View TTA)
import gc, math, os, random, threading, time

MODEL_INIT_LOCK = threading.Lock()
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def train_single_fold(
    k: int,
    spec: ModelSpec,
    tr_idx,
    va_idx,
    tr_bytes,
    te_bytes,
    y,
    bs: int,
    accum: int,
    eval_bs: int,
    epochs: int,
    eff_workers: int,
    views,
    cache_dir: str,
    resume: bool,
    device: torch.device,
    amp_dtype,
    use_scaler: bool,
    pretrained: bool,
    freeze_vit_blocks: int,
    p_nightmare: float,
    p_norm_view: float,
    seed: int,
    disable_grad_ckpt: bool = False,
    pre_probed=None,
    pin_memory=None,
):
    """Train one SGKF fold with epoch checkpoints, EMA tracking, and GPU 6-view OOF/ExtHard/Test."""
    t0 = time.time()
    if device.type == "cuda":
        torch.cuda.set_device(device)
    seed_everything(seed + k * 101)
    with MODEL_INIT_LOCK:
        model, mean_tup, std_tup, (n_train, n_tot) = build_model(
            spec,
            pretrained=pretrained,
            freeze_vit_blocks=freeze_vit_blocks,
            disable_grad_ckpt=disable_grad_ckpt,
        )
    model = model.to(device)
    if pre_probed is not None:
        use_cl, runtime_tag = pre_probed
        model.to(memory_format=(torch.channels_last if use_cl else torch.contiguous_format))
    else:
        use_cl, runtime_tag = probe_runtime_engine(model, spec, device, amp_dtype, use_scaler)
    seed_everything(seed + k * 101)

    mean_t = torch.tensor(mean_tup, device=device, dtype=torch.float32).view(1, 3, 1, 1)
    std_t = torch.tensor(std_tup, device=device, dtype=torch.float32).view(1, 3, 1, 1)

    pi_tr = np.bincount(y[tr_idx], minlength=5) / max(1, len(tr_idx))
    logit_adj = torch.tensor(0.20 * np.log(np.clip(pi_tr, 1e-4, 1.0)), device=device, dtype=torch.float32)

    eff_pin = (device.type == "cuda") if pin_memory is None else bool(pin_memory)
    tr_ds = Task4TrainDataset(tr_bytes, tr_idx, y, spec.img_hw, p_nightmare=p_nightmare, p_norm_view=p_norm_view, seed=seed + k * 1000)
    tr_dl_kw = {
        "batch_size": bs,
        "shuffle": True,
        "drop_last": len(tr_ds) >= bs,
        "num_workers": eff_workers,
        "pin_memory": eff_pin,
        "persistent_workers": (eff_workers > 0),
    }
    if eff_workers > 0:
        tr_dl_kw["prefetch_factor"] = 4
    tr_dl = DataLoader(tr_ds, **tr_dl_kw)

    opt = torch.optim.AdamW(make_param_groups(model, spec.base_lr, spec.backbone_lr_mult))
    total_steps = max(1, math.ceil(len(tr_dl) / accum)) * epochs
    warmup_steps = max(1, int(0.08 * total_steps))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt,
        lambda s: (s + 1) / warmup_steps if s < warmup_steps else 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * (s - warmup_steps) / max(1, total_steps - warmup_steps))),
    )
    scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)
    ema_state = {}

    gpu_tag = str(device)
    gc_tag = "off" if (disable_grad_ckpt or not spec.grad_ckpt) else "on"
    start_ep = 0
    ckpt = load_epoch_ckpt(cache_dir, spec, k, epochs, freeze_vit_blocks) if resume else None
    if ckpt is not None:
        model.load_state_dict(ckpt["trainable_model"], strict=False)
        for ek, ev in ckpt["ema_state"].items():
            ema_state[ek] = ev.to(device=device)
        opt.load_state_dict(ckpt["opt"])
        sched.load_state_dict(ckpt["sched"])
        if ckpt.get("scaler"):
            scaler.load_state_dict(ckpt["scaler"])
        start_ep = int(ckpt["epoch"]) + 1
        print(
            f"[{spec.key}] Fold {k} resumed ({gpu_tag}) Ep {start_ep}/{epochs} | "
            f"engine={runtime_tag} | amp={amp_dtype} | grad_ckpt={gc_tag} | trainable={n_train/1e6:.1f}M/{n_tot/1e6:.1f}M",
            flush=True,
        )
    else:
        print(
            f"[{spec.key}] Fold {k} started ({gpu_tag}) | engine={runtime_tag} | amp={amp_dtype} | "
            f"grad_ckpt={gc_tag} | trainable={n_train/1e6:.1f}M/{n_tot/1e6:.1f}M",
            flush=True,
        )

    for ep in range(start_ep, epochs):
        seed_everything(seed + k * 101 + ep * 17)
        model.train()
        opt.zero_grad(set_to_none=True)
        ep_loss, n_seen = 0.0, 0
        for step_i, (xb_u8, is_nm, yb) in enumerate(tr_dl):
            xb = xb_u8.to(device, non_blocking=eff_pin).float() * (1.0 / 255.0)
            nm_mask = is_nm.to(device=device, dtype=torch.bool, non_blocking=eff_pin)
            if nm_mask.any():
                xb[nm_mask] = apply_nightmare_photo_torch(xb[nm_mask])
            if p_norm_view > 0.0:
                gw_mask = torch.rand(xb.shape[0], device=device) < p_norm_view
                if gw_mask.any():
                    xb[gw_mask] = interior_gw_stretch_torch(xb[gw_mask])
            if use_cl and device.type == "cuda":
                xb = xb.to(memory_format=torch.channels_last)
            yb = yb.to(device, non_blocking=eff_pin)
            xb = (xb - mean_t) / std_t

            with torch.amp.autocast(device.type, dtype=amp_dtype) if amp_dtype is not None else torch.enable_grad():
                logits = model(xb)
                loss = F.cross_entropy(logits.float() + logit_adj, yb, label_smoothing=0.03)
                loss_scaled = loss / accum

            if use_scaler:
                scaler.scale(loss_scaled).backward()
            else:
                loss_scaled.backward()

            if ((step_i + 1) % accum == 0) or ((step_i + 1) == len(tr_dl)):
                if use_scaler:
                    scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), 2.0)
                if use_scaler:
                    prev_scale = scaler.get_scale()
                    scaler.step(opt)
                    scaler.update()
                    if scaler.get_scale() >= prev_scale:
                        sched.step()
                else:
                    opt.step()
                    sched.step()
                opt.zero_grad(set_to_none=True)
                update_ema(ema_state, model, decay=0.996 if ep >= 1 else 0.98)

            ep_loss += float(loss.item()) * len(yb)
            n_seen += len(yb)

        val_q = predict_views(model, tr_bytes, va_idx, spec, mean_t, std_t, ["orig"], eval_bs, device, amp_dtype, use_cl, True, eff_workers, pin_memory=eff_pin)["orig"]
        vf1 = f1_score(y[va_idx], val_q.argmax(1), average="macro")
        ckpt_saved = save_epoch_ckpt(cache_dir, spec, k, ep, epochs, freeze_vit_blocks, model, ema_state, opt, sched, scaler)
        if "/drive/" in ckpt_saved:
            ckpt_loc = "Drive:" + os.path.basename(ckpt_saved)
        elif "/kaggle/working" in ckpt_saved:
            ckpt_loc = "Working:" + os.path.basename(ckpt_saved)
        elif "/kaggle/temp" in ckpt_saved:
            ckpt_loc = "Temp:" + os.path.basename(ckpt_saved)
        else:
            ckpt_loc = "Local:" + os.path.basename(ckpt_saved)
        print(
            f"[{spec.key}] Fold {k} Ep {ep+1}/{epochs} ({gpu_tag}) loss={ep_loss/max(1, n_seen):.4f} "
            f"ValCleanF1={vf1:.5f} | saved ckpt ({ckpt_loc}) ({time.time()-t0:.1f}s)",
            flush=True,
        )

    model.load_state_dict(ema_state, strict=False)
    t_ev = time.time()
    va_views = predict_views(model, tr_bytes, va_idx, spec, mean_t, std_t, views, eval_bs, device, amp_dtype, use_cl, True, eff_workers, pin_memory=eff_pin)
    ext_imgs = make_deterministic_exthard(tr_bytes, va_idx, seed=123 + k)
    ext_views = predict_views(model, ext_imgs, None, spec, mean_t, std_t, views, eval_bs, device, amp_dtype, use_cl, False, eff_workers, pin_memory=eff_pin)
    te_views = predict_views(model, te_bytes, None, spec, mean_t, std_t, views, eval_bs, device, amp_dtype, use_cl, True, eff_workers, pin_memory=eff_pin)

    fold_out = {
        "va_orig2": combine_tta_views(va_views, "orig2"),
        "va_w6": combine_tta_views(va_views, "weighted6"),
        "ext_w6": combine_tta_views(ext_views, "weighted6"),
        "te_orig2": combine_tta_views(te_views, "orig2"),
        "te_w6": combine_tta_views(te_views, "weighted6"),
    }
    f_val = f1_score(y[va_idx], fold_out["va_w6"].argmax(1), average="macro")
    f_ext = f1_score(y[va_idx], fold_out["ext_w6"].argmax(1), average="macro")
    print(
        f"[{spec.key}] DONE Fold {k} ({gpu_tag}) in {time.time()-t0:.1f}s (eval={time.time()-t_ev:.1f}s) | "
        f"Val6TTA F1={f_val:.5f} | ExtHard F1={f_ext:.5f}",
        flush=True,
    )

    del model, opt, sched, scaler, ema_state, tr_dl, tr_ds, ext_imgs, va_views, ext_views, te_views
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return fold_out

print("Single-fold epoch-resumable trainer ready.")

In [ ]:
# Cell 10: 5-Fold SGKF Orchestrator (T4x2 Dual-GPU Split / H100 Single-GPU) & Fold/Bundle ZIP Packagers
import gc, glob, os, shutil, zipfile
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import torch
from sklearn.metrics import f1_score, log_loss

def package_model_zip(out_dir: str, model_key: str) -> str:
    """Package completed CSV/NPZ predictions for one backbone into a downloadable ZIP."""
    model_files = [
        f"task4-{model_key}-hard6-oof.csv",
        f"task4-{model_key}-hard6-test-predictions.csv",
        f"task4-{model_key}-orig2-oof.csv",
        f"task4-{model_key}-orig2-test-predictions.csv",
        f"task4-{model_key}-sgkf-hard-bundle.npz",
    ]
    mzip_out = os.path.join(out_dir, f"t4_hard_{model_key}.zip")
    with zipfile.ZipFile(mzip_out, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for fn in model_files:
            fp = os.path.join(out_dir, fn)
            if os.path.isfile(fp):
                zf.write(fp, fn)
    if os.path.isdir("/content") and not os.path.isdir("/kaggle/working") and os.path.abspath(mzip_out) != f"/content/t4_hard_{model_key}.zip":
        shutil.copy2(mzip_out, f"/content/t4_hard_{model_key}.zip")
    return mzip_out

def package_folds_zip(out_dir: str, model_key: str) -> str:
    """Package available fold{k}.npz checkpoints for partial-fold multi-machine merge."""
    cache_dir = os.path.join(out_dir, f"folds_{model_key}")
    fold_files = sorted(glob.glob(os.path.join(cache_dir, "fold*.npz"))) if os.path.isdir(cache_dir) else []
    fzip_out = os.path.join(out_dir, f"t4_hard_folds_{model_key}.zip")
    if not fold_files:
        return fzip_out
    with zipfile.ZipFile(fzip_out, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for fp in fold_files:
            zf.write(fp, os.path.join(f"folds_{model_key}", os.path.basename(fp)))
    return fzip_out

def train_one_model(
    spec: ModelSpec,
    tr_bytes,
    te_bytes,
    y,
    folds,
    tr_df,
    te_df,
    out_dir: str,
    device,
    pretrained=True,
    epochs_override=None,
    max_folds=5,
    folds_to_run=None,
    num_workers=4,
    resume=True,
    freeze_vit_blocks=None,
    p_nightmare=0.50,
    p_norm_view=0.25,
    seed=42,
    enable_dual_gpu=True,
    disable_grad_ckpt_high_vram=True,
):
    """Run or resume selected/all 5 SGKF folds (auto T4x2 parallel or H100 Turbo) and export ZIP."""
    os.makedirs(out_dir, exist_ok=True)
    cache_dir = os.path.join(out_dir, f"folds_{spec.key}")
    os.makedirs(cache_dir, exist_ok=True)

    amp_dtype, use_scaler = select_amp_dtype(device)
    bs, accum, eval_bs, disable_gc = pick_batch_and_accum(
        spec, device, target_eff_bs=48, disable_grad_ckpt_high_vram=disable_grad_ckpt_high_vram
    )
    epochs = epochs_override if epochs_override is not None else spec.epochs
    eff_freeze = spec.default_freeze_vit_blocks if freeze_vit_blocks is None else int(freeze_vit_blocks)
    eff_workers = min(num_workers, max(1, os.cpu_count() or num_workers)) if num_workers > 0 else 0
    views = ["orig", "flip", "gw", "gw_flip", "rot_p18_gw", "rot_m18_gw"]

    n_gpus = torch.cuda.device_count() if device.type == "cuda" else 0
    # Always inspect all 5 folds so attaching fold0.npz via /kaggle/input auto-completes full bundle
    fold_results = {}
    for k in range(5):
        fold_npz = os.path.join(cache_dir, f"fold{k}.npz")
        va_idx = np.where(folds == k)[0]
        cached = load_fold_npz(fold_npz) if resume else None
        if cached is not None:
            remove_epoch_ckpt(cache_dir, spec.key, k)
            fold_results[k] = cached
            print(
                f"[{spec.key}] Fold {k} resumed -> ValF1={f1_score(y[va_idx], cached['va_w6'].argmax(1), average='macro'):.5f} | "
                f"ExtHardF1={f1_score(y[va_idx], cached['ext_w6'].argmax(1), average='macro'):.5f}"
            )

    requested_folds = [int(k) for k in folds_to_run] if folds_to_run is not None else list(range(min(5, max_folds)))
    pending_folds = [k for k in requested_folds if k not in fold_results]

    use_dual_gpu = bool(enable_dual_gpu and n_gpus >= 2 and len(pending_folds) >= 2)
    print(
        f"[{spec.key}] Config: img_hw={spec.img_hw} | bs={bs} (accum={accum}, eval_bs={eval_bs}) | "
        f"epochs={epochs} | amp={amp_dtype} | grad_ckpt={not disable_gc if spec.grad_ckpt else False} | "
        f"freeze_vit_blocks={eff_freeze} | workers={0 if use_dual_gpu else eff_workers} | "
        f"dual_gpu={use_dual_gpu} | requested={requested_folds} | pending={pending_folds}"
    )

    if use_dual_gpu:
        probe_dev = torch.device("cuda:0")
        probe_m, _, _, _ = build_model(
            spec, pretrained=pretrained, freeze_vit_blocks=eff_freeze, disable_grad_ckpt=disable_gc
        )
        probe_m = probe_m.to(probe_dev)
        pre_probed = probe_runtime_engine(probe_m, spec, probe_dev, amp_dtype, use_scaler)
        del probe_m
        gc.collect()
        torch.cuda.empty_cache()

        def _gpu_worker(gpu_id: int, fold_subset):
            dev_g = torch.device(f"cuda:{gpu_id}")
            amp_g, sc_g = select_amp_dtype(dev_g)
            bs_g, acc_g, ev_g, dgc_g = pick_batch_and_accum(
                spec, dev_g, target_eff_bs=48, disable_grad_ckpt_high_vram=disable_grad_ckpt_high_vram
            )
            local_out = {}
            for k in fold_subset:
                va_idx, tr_idx = np.where(folds == k)[0], np.where(folds != k)[0]
                fold_npz = os.path.join(cache_dir, f"fold{k}.npz")
                res = train_single_fold(
                    k=k, spec=spec, tr_idx=tr_idx, va_idx=va_idx, tr_bytes=tr_bytes, te_bytes=te_bytes, y=y,
                    bs=bs_g, accum=acc_g, eval_bs=ev_g, epochs=epochs, eff_workers=0, views=views,
                    cache_dir=cache_dir, resume=resume, device=dev_g, amp_dtype=amp_g, use_scaler=sc_g,
                    pretrained=pretrained, freeze_vit_blocks=eff_freeze, p_nightmare=p_nightmare,
                    p_norm_view=p_norm_view, seed=seed, disable_grad_ckpt=dgc_g, pre_probed=pre_probed, pin_memory=False,
                )
                atomic_savez_compressed(fold_npz, **res)
                remove_epoch_ckpt(cache_dir, spec.key, k)
                local_out[k] = res
            return local_out

        with ThreadPoolExecutor(max_workers=2) as pool:
            futs = [pool.submit(_gpu_worker, gid, pending_folds[gid::2]) for gid in range(2) if pending_folds[gid::2]]
            for fut in futs:
                fold_results.update(fut.result())
    else:
        for k in pending_folds:
            va_idx, tr_idx = np.where(folds == k)[0], np.where(folds != k)[0]
            fold_npz = os.path.join(cache_dir, f"fold{k}.npz")
            res = train_single_fold(
                k=k, spec=spec, tr_idx=tr_idx, va_idx=va_idx, tr_bytes=tr_bytes, te_bytes=te_bytes, y=y,
                bs=bs, accum=accum, eval_bs=eval_bs, epochs=epochs, eff_workers=eff_workers, views=views,
                cache_dir=cache_dir, resume=resume, device=device, amp_dtype=amp_dtype, use_scaler=use_scaler,
                pretrained=pretrained, freeze_vit_blocks=eff_freeze, p_nightmare=p_nightmare,
                p_norm_view=p_norm_view, seed=seed, disable_grad_ckpt=disable_gc,
            )
            atomic_savez_compressed(fold_npz, **res)
            remove_epoch_ckpt(cache_dir, spec.key, k)
            fold_results[k] = res

    if len(fold_results) == 5 and all(k in fold_results for k in range(5)):
        n_tr, n_te = len(tr_df), len(te_df)
        oof_orig2 = np.zeros((n_tr, 5), dtype=np.float32)
        oof_w6 = np.zeros((n_tr, 5), dtype=np.float32)
        ext_w6 = np.zeros((n_tr, 5), dtype=np.float32)
        te_orig2 = np.zeros((n_te, 5), dtype=np.float32)
        te_w6 = np.zeros((n_te, 5), dtype=np.float32)
        for k in range(5):
            va_idx = np.where(folds == k)[0]
            fr = fold_results[k]
            oof_orig2[va_idx], oof_w6[va_idx], ext_w6[va_idx] = fr["va_orig2"], fr["va_w6"], fr["ext_w6"]
            te_orig2 += fr["te_orig2"] / 5.0
            te_w6 += fr["te_w6"] / 5.0

        f_all = f1_score(y, oof_w6.argmax(1), average="macro")
        f_o2 = f1_score(y, oof_orig2.argmax(1), average="macro")
        f_ext = f1_score(y, ext_w6.argmax(1), average="macro")
        ll_all = log_loss(y, np.clip(oof_w6, 1e-7, 1.0), labels=[0, 1, 2, 3, 4])
        print(
            f"\n=== [{spec.key}] 5-FOLD COMPLETE | OOF_W6 F1={f_all:.5f} LL={ll_all:.5f} | "
            f"Orig2 F1={f_o2:.5f} | ExtHard F1={f_ext:.5f} | Combo={0.5*(f_all+f_ext):.5f} ===\n",
            flush=True,
        )
        save_csv_preds(os.path.join(out_dir, f"task4-{spec.key}-hard6-oof.csv"), tr_df, oof_w6, folds)
        save_csv_preds(os.path.join(out_dir, f"task4-{spec.key}-hard6-test-predictions.csv"), te_df, te_w6)
        save_csv_preds(os.path.join(out_dir, f"task4-{spec.key}-orig2-oof.csv"), tr_df, oof_orig2, folds)
        save_csv_preds(os.path.join(out_dir, f"task4-{spec.key}-orig2-test-predictions.csv"), te_df, te_orig2)
        atomic_savez_compressed(
            os.path.join(out_dir, f"task4-{spec.key}-sgkf-hard-bundle.npz"),
            folds=folds, oof_orig2=oof_orig2, oof_w6=oof_w6, ext_w6=ext_w6, te_orig2=te_orig2, te_w6=te_w6,
        )
        mzip = package_model_zip(out_dir, spec.key)
        print(f"[READY] Saved 5-fold {spec.key} zip -> {mzip}", flush=True)
    else:
        fzip = package_folds_zip(out_dir, spec.key)
        print(f"[PARTIAL READY] Completed folds {sorted(fold_results.keys())}/5 | Saved fold zip -> {fzip}", flush=True)

print("5-fold orchestrator (T4x2 Dual-GPU + Partial-Fold + H100 Turbo) ready.")

In [ ]:
# Cell 11: Preload Manifest + JPEG Buffers & Run Selected Backbones Live
import os, time
import numpy as np
import pandas as pd

tr_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
te_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
if "file_name" not in tr_df.columns:
    tr_df["file_name"] = tr_df["id"].astype(str) + ".jpg"
if "file_name" not in te_df.columns:
    te_df["file_name"] = te_df["id"].astype(str) + ".jpg"

cmap = {"7-11k": 0, "7-11K": 0, "fiveN": 1, "freefried": 2, "pube": 3, "roblock": 4}
y = tr_df["label"].astype(int).to_numpy() if "label" in tr_df.columns else tr_df["class"].map(cmap).astype(int).to_numpy()
folds = np.load(os.path.join(DATA_DIR, "folds_sgkf.npy")).astype(int)
assert len(tr_df) == len(folds) == len(y)

print("Preloading compressed JPEG bytes into RAM...")
t_load = time.time()
tr_bytes = load_jpeg_bytes(os.path.join(DATA_DIR, "train"), tr_df["file_name"].tolist())
te_bytes = load_jpeg_bytes(os.path.join(DATA_DIR, "test"), te_df["file_name"].tolist())
print(f"Preloaded train={len(tr_bytes):,}, test={len(te_bytes):,} images in {time.time()-t_load:.1f}s")

for mkey in MODELS_TO_RUN:
    vit_freeze = FREEZE_VIT_BLOCKS if (FREEZE_T4_LOWER and FREEZE_VIT_BLOCKS is not None) else (None if FREEZE_T4_LOWER else 0)
    print(f"\n>>> Starting backbone [{mkey}] ({MODEL_SPECS[mkey].timm_name}) | checkpoints: {OUT_DIR}/folds_{mkey}")
    train_one_model(
        spec=MODEL_SPECS[mkey],
        tr_bytes=tr_bytes,
        te_bytes=te_bytes,
        y=y,
        folds=folds,
        tr_df=tr_df,
        te_df=te_df,
        out_dir=OUT_DIR,
        device=DEVICE,
        pretrained=True,
        epochs_override=EPOCHS_OVERRIDE,
        max_folds=MAX_FOLDS,
        folds_to_run=FOLDS_TO_RUN,
        num_workers=NUM_WORKERS,
        resume=True,
        freeze_vit_blocks=vit_freeze,
        p_nightmare=P_NIGHTMARE,
        p_norm_view=P_NORM_VIEW,
        seed=SEED,
        enable_dual_gpu=ENABLE_DUAL_GPU_PARALLEL,
        disable_grad_ckpt_high_vram=DISABLE_GRAD_CKPT_HIGH_VRAM,
    )

In [ ]:
# Cell 12: Verify All Completed Bundles & Fold Checkpoints -> Package t4_hard_bundles.zip
import glob, os, re, shutil, zipfile
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, log_loss

bundle_files = sorted(glob.glob(os.path.join(OUT_DIR, "task4-*.csv")) + glob.glob(os.path.join(OUT_DIR, "task4-*.npz")))
fold_ckpts = sorted(glob.glob(os.path.join(OUT_DIR, "folds_*", "fold*.npz")))
assert bundle_files or fold_ckpts, f"No completed bundles or fold*.npz checkpoints found in {OUT_DIR} yet!"

def _load_y_and_folds():
    if "y" in globals() and "folds" in globals():
        return y, folds
    tr_meta = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
    cmap_fb = {"7-11k": 0, "7-11K": 0, "fiveN": 1, "freefried": 2, "pube": 3, "roblock": 4}
    y_loc = tr_meta["label"].astype(int).to_numpy() if "label" in tr_meta.columns else tr_meta["class"].map(cmap_fb).astype(int).to_numpy()
    f_loc = np.load(os.path.join(DATA_DIR, "folds_sgkf.npy")).astype(int)
    return y_loc, f_loc

zip_path = os.path.join(OUT_DIR, "t4_hard_bundles.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for fn in bundle_files:
        zf.write(fn, os.path.basename(fn))
        if fn.endswith("hard6-oof.csv"):
            df_o = pd.read_csv(fn)
            if "label" in df_o.columns:
                y_eval = df_o["label"].astype(int).to_numpy()
            else:
                y_eval, _ = _load_y_and_folds()
            p_o = df_o[[f"prob_{i}" for i in range(5)]].to_numpy()
            p_o = p_o / np.clip(p_o.sum(axis=1, keepdims=True), 1e-12, None)
            f_clean = f1_score(y_eval, p_o.argmax(1), average="macro")
            ll_val = log_loss(y_eval, np.clip(p_o, 1e-7, 1.0), labels=[0, 1, 2, 3, 4])
            npz_match = fn.replace("-hard6-oof.csv", "-sgkf-hard-bundle.npz")
            ext_str = ""
            if os.path.isfile(npz_match):
                with np.load(npz_match) as z:
                    if "ext_w6" in z:
                        f_ext = f1_score(y_eval, z["ext_w6"].argmax(1), average="macro")
                        ext_str = f" | ExtHard F1={f_ext:.6f} | Combo={0.5*(f_clean+f_ext):.6f}"
            print(f"{os.path.basename(fn):40s} | OOF_W6 F1={f_clean:.6f} | LL={ll_val:.6f}{ext_str}")

    if fold_ckpts:
        y_all, folds_all = _load_y_and_folds()
        print("\n--- Individual Fold Checkpoints Included ---")
        for fp in fold_ckpts:
            rel = os.path.relpath(fp, OUT_DIR)
            zf.write(fp, rel)
            m = re.search(r"fold(\d+)\.npz$", os.path.basename(fp))
            if m:
                k = int(m.group(1))
                va_idx = np.where(folds_all == k)[0]
                with np.load(fp) as z:
                    if "va_w6" in z and len(z["va_w6"]) == len(va_idx):
                        f_v = f1_score(y_all[va_idx], z["va_w6"].argmax(1), average="macro")
                        f_e = f1_score(y_all[va_idx], z["ext_w6"].argmax(1), average="macro")
                        print(f"{rel:40s} | n={len(va_idx)} | Val6TTA F1={f_v:.6f} | ExtHard F1={f_e:.6f}")

print("Packaged combined download bundle ->", zip_path, f"({os.path.getsize(zip_path)/1024**2:.2f} MB)")
if os.path.isdir("/content") and not os.path.isdir("/kaggle/working") and os.path.abspath(zip_path) != "/content/t4_hard_bundles.zip":
    shutil.copy2(zip_path, "/content/t4_hard_bundles.zip")
    print("Copied -> /content/t4_hard_bundles.zip")